In [104]:
import pandas as pd
import numpy as np
from pathlib import Path

In [105]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [106]:
# 2. Load the original dataset
DATA_PATH = Path(
    "/content/drive/MyDrive/SEM-6/Mini-Project/dataset.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}\n"
        "Check the file path in Google Drive."
    )

df_original = pd.read_csv(DATA_PATH)

In [107]:
# 3. Basic dataset information
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print("Number of rows:", df_original.shape[0])
print("Number of columns:", df_original.shape[1])
print("\nColumn names:")
print(df_original.columns.tolist())

print("\nData types:")
print(df_original.dtypes.to_string())

DATASET OVERVIEW
Number of rows: 3400
Number of columns: 11

Column names:
['scheme_name', 'slug', 'details', 'benefits', 'eligibility', 'application', 'documents', 'level', 'schemeCategory', 'Unnamed: 9', 'tags']

Data types:
scheme_name        object
slug               object
details            object
benefits           object
eligibility        object
application        object
documents          object
level              object
schemeCategory     object
Unnamed: 9        float64
tags               object


In [108]:
# 4. Inspect sample records
print("\nFIRST 5 RECORDS")
display(df_original.head())


FIRST 5 RECORDS


,scheme_name,slug,details,benefits,eligibility,application,documents,level,schemeCategory,Unnamed: 9,tags
0,"""Immediate Relief Assistance"" under ""Welfare a...",ira-wrflsncs,"The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",The applicant should be the family (legal heir...,Step 1: The interested applicant should visit ...,Photograph of the Family (Legal Heir) of the M...,State,"Agriculture,Rural & Environment, Social welfar...",NaN,"Missing, Fisherman, Relief, Financial Assistan..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,"Short Term Training Programme-SFURTI Program, ...","Financial Assistance : Limit of funding ₹ 4,00...",The institution should be AICTE approved.,Registration of New Institute: Step 01: Visit ...,Feedback Form Copy of Proceedings Completion R...,Central,Education & Learning,NaN,"Trainings, Financial Assistance, AICTE"
2,Burial and Ex-gratia Payment Scheme in Case of...,baepsicodouldwact,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",The deceased construction worker should have b...,Step 1: The interested applicant should visit ...,Aadhaar Card of the applicant (nominee/Legal h...,State,Social welfare & Empowerment,NaN,"Building Worker, Construction Workers, Unregis..."
3,Consortia & Tender Marketing Scheme,ctms,Promotion of the product of Micro and Small En...,Enlistment of the Unit for participating in Go...,Micro & Small Enterprises registered with NSIC...,"Step 01: The application form, in the prescrib...",A passport size photograph of each of the Prop...,Central,Business & Entrepreneurship,NaN,"Goods And Services Marketing, Turnkey Projects..."
4,Garuda Scheme for Funeral Expense,gsfe,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",The applicant should be a close relative of th...,Registration and apply Step 01: Applicants hav...,Passport-size Photograph of the Applicant Aadh...,State,Social welfare & Empowerment,NaN,"Social Welfare, Financial Assistance, Deceased..."


In [109]:
# 5. Missing values
print("\nMISSING VALUES")
missing_report = pd.DataFrame({
    "missing_count": df_original.isna().sum(),
    "missing_percent": (
        df_original.isna().mean() * 100
    ).round(2),
    "data_type": df_original.dtypes.astype(str)
})
display(missing_report.sort_values(
    "missing_count", ascending=False
))


MISSING VALUES


,missing_count,missing_percent,data_type
Unnamed: 9,3400,100.00,float64
tags,29,0.85,object
documents,11,0.32,object
application,2,0.06,object
details,0,0.00,object
scheme_name,0,0.00,object
slug,0,0.00,object
benefits,0,0.00,object
eligibility,0,0.00,object
schemeCategory,0,0.00,object


In [110]:
# 6. Duplicate checks
print("\nDUPLICATE CHECKS")
print(
    "Fully duplicated rows:",
    int(df_original.duplicated().sum())
)

if "scheme_name" in df_original.columns:
    names = df_original["scheme_name"].dropna().astype(str).str.strip()
    print(
        "Repeated non-empty scheme names:",
        int(names.str.casefold().duplicated().sum())
    )
else:
    print("WARNING: scheme_name column was not found.")



DUPLICATE CHECKS
Fully duplicated rows: 3
Repeated non-empty scheme names: 3


In [111]:
# 7. Inspect category distribution
if "schemeCategory" in df_original.columns:
    print("\nCATEGORY DISTRIBUTION")
    print(
        df_original["schemeCategory"]
        .fillna("[MISSING]")
        .astype(str)
        .value_counts()
        .to_string()
    )
else:
    print("WARNING: schemeCategory column was not found.")


CATEGORY DISTRIBUTION
schemeCategory
Social welfare & Empowerment                                                                                                                                                    641
Education & Learning                                                                                                                                                            558
Agriculture,Rural & Environment                                                                                                                                                 386
Business & Entrepreneurship                                                                                                                                                     332
Social welfare & Empowerment, Women and Child                                                                                                                                   149
Health & Wellness                                             

In [112]:
# 8. Check expected text fields
expected_text_columns = [
    "scheme_name",
    "details",
    "benefits",
    "eligibility",
    "application",
    "documents",
    "tags",
    "schemeCategory"
]

print("\nEXPECTED COLUMN CHECK")
for col in expected_text_columns:
    print(f"{col}: {'FOUND' if col in df_original.columns else 'MISSING'}")

print("\nStep 1 inspection completed. Original dataset was not modified.")



EXPECTED COLUMN CHECK
scheme_name: FOUND
details: FOUND
benefits: FOUND
eligibility: FOUND
application: FOUND
documents: FOUND
tags: FOUND
schemeCategory: FOUND

Step 1 inspection completed. Original dataset was not modified.


In [113]:
import pandas as pd
import numpy as np
import re

In [114]:
# Start from the original dataset loaded in Step 1
df_work = df_original.copy()

In [115]:
# 1. Preserve original row position for traceability
df_work.insert(
    0,
    "source_row_id",
    np.arange(len(df_work), dtype=int)
)


In [116]:
# 2. Remove columns that contain no useful information
empty_columns = [
    col for col in df_work.columns
    if df_work[col].isna().all()
]
df_work = df_work.drop(columns=empty_columns)

print("Removed completely empty columns:", empty_columns)

Removed completely empty columns: ['Unnamed: 9']


In [117]:
# 3. Identify columns that should contain text
text_columns = [
    "scheme_name",
    "slug",
    "details",
    "benefits",
    "eligibility",
    "application",
    "documents",
    "level",
    "schemeCategory",
    "tags"
]

# Stop if required columns are missing
required_columns = [
    "scheme_name",
    "slug",
    "details",
    "benefits",
    "eligibility",
    "schemeCategory"
]

missing_required = [
    col for col in required_columns
    if col not in df_work.columns
]

if missing_required:
    raise ValueError(
        f"Required columns are missing: {missing_required}"
    )


In [118]:
# 4. Fill missing text safely and normalize whitespace
for col in text_columns:
    if col in df_work.columns:
        df_work[col] = (
            df_work[col]
            .fillna("")
            .astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )


In [119]:
# 5. Check essential fields that are empty after cleaning
print("\nEMPTY ESSENTIAL FIELDS")
for col in required_columns:
    print(
        f"{col}:",
        int(df_work[col].eq("").sum())
    )


EMPTY ESSENTIAL FIELDS
scheme_name: 0
slug: 0
details: 0
benefits: 0
eligibility: 0
schemeCategory: 0


In [120]:
# 6. Inspect fully duplicated records (excluding source row ID)
comparison_columns = [
    col for col in df_work.columns
    if col != "source_row_id"
]

duplicate_mask = df_work.duplicated(
    subset=comparison_columns,
    keep=False
)

print("\nFULL DUPLICATE RECORDS")
print("Rows belonging to duplicate groups:", int(duplicate_mask.sum()))

if duplicate_mask.any():
    display(
        df_work.loc[
            duplicate_mask,
            ["source_row_id", "scheme_name", "slug", "schemeCategory"]
        ].sort_values("slug").head(30)
    )



FULL DUPLICATE RECORDS
Rows belonging to duplicate groups: 5


,source_row_id,scheme_name,slug,schemeCategory
898,898,Establishment of Goat Unit (10 +1),eogu,"Agriculture,Rural & Environment"
899,899,Establishment of Goat Unit (10 +1),eogu,"Agriculture,Rural & Environment"
900,900,Establishment of Goat Unit (10 +1),eogu,"Agriculture,Rural & Environment"
3270,3270,Valmiki Chhatravritti Yojana,vcy,Education & Learning
3271,3271,Valmiki Chhatravritti Yojana,vcy,Education & Learning


In [121]:
# 7. Inspect repeated scheme names
name_mask = (
    df_work["scheme_name"].ne("")
    & df_work["scheme_name"].str.casefold().duplicated(keep=False)
)

print("\nREPEATED SCHEME NAMES")
print("Rows with repeated names:", int(name_mask.sum()))

if name_mask.any():
    display(
        df_work.loc[
            name_mask,
            [
                "source_row_id", "scheme_name", "slug",
                "level", "schemeCategory"
            ]
        ].sort_values("scheme_name").head(50)
    )



REPEATED SCHEME NAMES
Rows with repeated names: 5


,source_row_id,scheme_name,slug,level,schemeCategory
898,898,Establishment of Goat Unit (10 +1),eogu,State,"Agriculture,Rural & Environment"
899,899,Establishment of Goat Unit (10 +1),eogu,State,"Agriculture,Rural & Environment"
900,900,Establishment of Goat Unit (10 +1),eogu,State,"Agriculture,Rural & Environment"
3270,3270,Valmiki Chhatravritti Yojana,vcy,State,Education & Learning
3271,3271,Valmiki Chhatravritti Yojana,vcy,State,Education & Learning


In [122]:
# 8. Check slug uniqueness
print("\nSLUG CHECK")
print("Empty slugs:", int(df_work["slug"].eq("").sum()))
print(
    "Repeated non-empty slugs:",
    int(
        df_work.loc[df_work["slug"].ne(""), "slug"]
        .duplicated(keep=False).sum()
    )
)



SLUG CHECK
Empty slugs: 0
Repeated non-empty slugs: 5


In [123]:
# 9. Check category formatting
print("\nCATEGORY CHECK")
print("Empty categories:", int(df_work["schemeCategory"].eq("").sum()))
print("Distinct raw category strings:", df_work["schemeCategory"].nunique())

print("\nSample category values:")
display(
    df_work["schemeCategory"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
    .head(20)
)



CATEGORY CHECK
Empty categories: 0
Distinct raw category strings: 210

Sample category values:


,category,count
0,Social welfare & Empowerment,641
1,Education & Learning,558
2,"Agriculture,Rural & Environment",386
3,Business & Entrepreneurship,332
4,"Social welfare & Empowerment, Women and Child",149
5,Health & Wellness,100
6,Skills & Employment,94
7,"Education & Learning, Social welfare & Empower...",75
8,"Banking,Financial Services and Insurance",58
9,Housing & Shelter,47


In [124]:
# 10. Verify the working dataset
print("\nFINAL STEP 2 SUMMARY")
print("Rows:", len(df_work))
print("Columns:", len(df_work.columns))
print("Original dataset rows:", len(df_original))
print(
    "Source row IDs preserved:",
    df_work["source_row_id"].is_unique
)
print("Original dataframe unchanged:", len(df_original) == 3400)



FINAL STEP 2 SUMMARY
Rows: 3400
Columns: 11
Original dataset rows: 3400
Source row IDs preserved: True
Original dataframe unchanged: True


In [125]:
import pandas as pd
import numpy as np

# Start with the Step 2 dataframe
df_step3 = df_work.copy()

In [126]:
# --------------------------------------------------
# 1. Remove exact duplicate records
#    Ignore source_row_id when comparing records.
# --------------------------------------------------
comparison_columns = [
    col for col in df_step3.columns
    if col != "source_row_id"
]

before = len(df_step3)

df_step3 = df_step3.drop_duplicates(
    subset=comparison_columns,
    keep="first"
).copy()

df_step3 = df_step3.reset_index(drop=True)

print("Rows before deduplication:", before)
print("Rows after deduplication:", len(df_step3))
print("Duplicate rows removed:", before - len(df_step3))


Rows before deduplication: 3400
Rows after deduplication: 3397
Duplicate rows removed: 3


In [127]:
# --------------------------------------------------
# 2. Check slug collisions after deduplication
# --------------------------------------------------
slug_counts = df_step3["slug"].value_counts()
duplicate_slugs = slug_counts[slug_counts > 1]

print("\nRemaining duplicate slugs:", len(duplicate_slugs))

if len(duplicate_slugs) > 0:
    print("Review these slugs before proceeding:")
    print(duplicate_slugs.to_string())
else:
    print("All slugs are unique after deduplication.")



Remaining duplicate slugs: 0
All slugs are unique after deduplication.


In [128]:
# --------------------------------------------------
# 3. Create scheme_id
#    Use slug as the stable ID only if unique.
#    Otherwise, create collision-safe IDs and report them.
# --------------------------------------------------
if df_step3["slug"].is_unique:
    df_step3["scheme_id"] = "SCHEME_" + df_step3["slug"].astype(str)
else:
    # Preserve every record; do not merge different schemes
    # merely because their slugs collide.
    df_step3["scheme_id"] = (
        "SCHEME_ROW_" +
        df_step3["source_row_id"].astype(str)
    )

print("\nUnique scheme IDs:", df_step3["scheme_id"].is_unique)
print("Scheme ID sample:")
print(
    df_step3[
        ["source_row_id", "scheme_id", "scheme_name", "slug"]
    ].head(10).to_string(index=False)
)



Unique scheme IDs: True
Scheme ID sample:
 source_row_id                scheme_id                                                                                                              scheme_name              slug
             0      SCHEME_ira-wrflsncs "Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme"      ira-wrflsncs
             1            SCHEME_astpss                                                                        AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME            astpss
             2 SCHEME_baepsicodouldwact            Burial and Ex-gratia Payment Scheme in Case of Death of Unregistered Laborer During Work at Construction Site baepsicodouldwact
             3              SCHEME_ctms                                                                                      Consortia & Tender Marketing Scheme              ctms
             4              SCHEME_gsfe                       

In [129]:
# --------------------------------------------------
# 4. Build retrieval text
#    Keep natural wording for sentence-transformer embeddings.
# --------------------------------------------------
retrieval_fields = [
    ("Scheme", "scheme_name"),
    ("Description", "details"),
    ("Benefits", "benefits"),
    ("Eligibility", "eligibility"),
    ("Application", "application"),
    ("Required documents", "documents"),
    ("Level", "level"),
    ("Categories", "schemeCategory"),
    ("Keywords", "tags"),
]

def make_retrieval_text(row):
    sections = []

    for label, column in retrieval_fields:
        if column not in row.index:
            continue

        value = row[column]

        if pd.isna(value):
            continue

        value = str(value).strip()

        if value:
            sections.append(f"{label}: {value}")

    return "\n".join(sections)

df_step3["retrieval_text"] = df_step3.apply(
    make_retrieval_text,
    axis=1
)

In [130]:
# --------------------------------------------------
# 5. Validate the prepared dataset
# --------------------------------------------------
print("\nRETRIEVAL TEXT CHECK")
print("Empty retrieval texts:", int(
    df_step3["retrieval_text"].str.strip().eq("").sum()
))
print("Unique scheme IDs:", df_step3["scheme_id"].nunique())
print("Dataset rows:", len(df_step3))

print("\nExample retrieval text:")
print(df_step3.iloc[0]["retrieval_text"][:1500])


RETRIEVAL TEXT CHECK
Empty retrieval texts: 0
Unique scheme IDs: 3397
Dataset rows: 3397

Example retrieval text:
Scheme: "Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme"
Description: The scheme "Immediate Relief Assistance" is a Sub-Component under the scheme "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme". The scheme is extended to all the regions of the Union territory of Puducherry. The scheme is introduced with the objective of extending financial assistance to the fishermen's families to compensate for the loss due to the missing breadwinner and to support them financially to run their family.
Benefits: ₹ 1,00,000, in two installments of ₹ 50,000 each, as immediate relief assistance for the family (legal heir) of the missing fisherman. ﻿ Disbursal Initially, 50% will be extended within 3 months from the date of receipt of the application from the family (legal heir). The 

In [131]:
# --------------------------------------------------
# 6. Save a new processed dataset
# --------------------------------------------------
OUTPUT_PATH = "processed_schemes_v2.csv"

df_step3.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved: {OUTPUT_PATH}")
print("Original dataset was not overwritten.")



Saved: processed_schemes_v2.csv
Original dataset was not overwritten.


In [132]:
import pandas as pd
import re
import json

# Load the output from Step 3
df_step4 = pd.read_csv("processed_schemes_v2.csv")


In [133]:
# 1. Define the canonical category labels from your dataset
canonical_categories = [
    "Education & Learning",
    "Skills & Employment",
    "Health & Wellness",
    "Social welfare & Empowerment",
    "Business & Entrepreneurship",
    "Banking,Financial Services and Insurance",
    "Women and Child",
    "Housing & Shelter",
    "Utility & Sanitation",
    "Transport & Infrastructure",
    "Science, IT & Communications",
    "Sports & Culture",
    "Travel & Tourism",
    "Public Safety,Law & Justice",
    "Agriculture,Rural & Environment",
]

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", str(text)).strip().casefold()

# Match longer labels first, without splitting on commas.
category_lookup = {
    normalize_whitespace(category): category
    for category in canonical_categories
}


In [134]:
# 2. Identify categories by matching complete known labels.
def extract_categories(raw_value):
    raw = normalize_whitespace(raw_value)

    if not raw:
        return []

    found = [
        category
        for normalized, category in category_lookup.items()
        if normalized in raw
    ]

    return sorted(set(found))

df_step4["category_list"] = df_step4["schemeCategory"].apply(
    extract_categories
)



In [135]:
# 3. Flag values that contain no recognized categories.
df_step4["category_unrecognized"] = (
    df_step4["category_list"].str.len() == 0
)

In [136]:
# 4. Add a compact normalized category field for filtering.
df_step4["normalized_categories"] = df_step4["category_list"].apply(
    lambda categories: " | ".join(categories)
)

In [137]:
# 5. Report category extraction results.
print("ROWS:", len(df_step4))
print(
    "Rows with no recognized categories:",
    int(df_step4["category_unrecognized"].sum())
)

print("\nCATEGORY FREQUENCY")
category_counts = (
    df_step4["category_list"]
    .explode()
    .dropna()
    .value_counts()
)
print(category_counts.to_string())

print("\nNUMBER OF CATEGORIES PER SCHEME")
print(
    df_step4["category_list"]
    .str.len()
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nEXAMPLES")
display(
    df_step4[
        [
            "scheme_id",
            "scheme_name",
            "schemeCategory",
            "category_list",
            "normalized_categories",
            "category_unrecognized",
        ]
    ].head(15)
)


ROWS: 3397
Rows with no recognized categories: 0

CATEGORY FREQUENCY
category_list
Social welfare & Empowerment                1281
Education & Learning                         833
Business & Entrepreneurship                  515
Agriculture,Rural & Environment              479
Women and Child                              376
Skills & Employment                          265
Banking,Financial Services and Insurance     223
Health & Wellness                            218
Sports & Culture                             118
Housing & Shelter                             92
Science, IT & Communications                  69
Transport & Infrastructure                    54
Utility & Sanitation                          37
Travel & Tourism                              36
Public Safety,Law & Justice                   10

NUMBER OF CATEGORIES PER SCHEME
category_list
1    2354
2     903
3     120
4      16
5       2
6       2

EXAMPLES


,scheme_id,scheme_name,schemeCategory,category_list,normalized_categories,category_unrecognized
0,SCHEME_ira-wrflsncs,"""Immediate Relief Assistance"" under ""Welfare a...","Agriculture,Rural & Environment, Social welfar...","[Agriculture,Rural & Environment, Social welfa...","Agriculture,Rural & Environment | Social welfa...",False
1,SCHEME_astpss,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,Education & Learning,[Education & Learning],Education & Learning,False
2,SCHEME_baepsicodouldwact,Burial and Ex-gratia Payment Scheme in Case of...,Social welfare & Empowerment,[Social welfare & Empowerment],Social welfare & Empowerment,False
3,SCHEME_ctms,Consortia & Tender Marketing Scheme,Business & Entrepreneurship,[Business & Entrepreneurship],Business & Entrepreneurship,False
4,SCHEME_gsfe,Garuda Scheme for Funeral Expense,Social welfare & Empowerment,[Social welfare & Empowerment],Social welfare & Empowerment,False
5,SCHEME_ifticmwstc,Incentive For The Intra Caste Marriage within ...,Social welfare & Empowerment,[Social welfare & Empowerment],Social welfare & Empowerment,False
6,SCHEME_ismpsscis,Incentive Scheme for MSMEs in Powerloom Sector...,Business & Entrepreneurship,[Business & Entrepreneurship],Business & Entrepreneurship,False
7,SCHEME_imspesy,Indira Mahila Shakti Udyam Protsahan Yojana,Women and Child,[Women and Child],Women and Child,False
8,SCHEME_msasy,Mukhyamantri Shramik Aujaar Sahayata Yojana,Social welfare & Empowerment,[Social welfare & Empowerment],Social welfare & Empowerment,False
9,SCHEME_nsjbsy,Nirman Shramik Jeevan va Bhavishya Suraksha Yo...,Health & Wellness,[Health & Wellness],Health & Wellness,False


In [138]:
# 6. Inspect any unrecognized values before deciding how to handle them.
unrecognized = (
    df_step4.loc[
        df_step4["category_unrecognized"],
        "schemeCategory"
    ]
    .value_counts()
)

print("\nUNRECOGNIZED CATEGORY VALUES")
if unrecognized.empty:
    print("None")
else:
    print(unrecognized.to_string())


UNRECOGNIZED CATEGORY VALUES
None


In [139]:
# 7. Save a new version; preserve all rows for now.
df_step4.to_csv("processed_schemes_v3.csv", index=False)

print("\nSaved processed_schemes_v3.csv")



Saved processed_schemes_v3.csv


In [140]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Load the validated dataset from Step 4
df_step5 = pd.read_csv("processed_schemes_v3.csv")

In [141]:
# 1. Verify required columns
required_columns = [
    "source_row_id",
    "scheme_id",
    "scheme_name",
    "slug",
    "details",
    "benefits",
    "eligibility",
    "application",
    "documents",
    "level",
    "schemeCategory",
    "category_list",
    "normalized_categories",
    "retrieval_text",
]

missing_columns = [
    col for col in required_columns
    if col not in df_step5.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


In [142]:
# 2. Parse category_list safely.
# CSV may store Python-style list strings, so use JSON only
# after validating the actual format.
def parse_category_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value) or not str(value).strip():
        return []

    # The saved CSV typically represents lists as:
    # ['Education & Learning', 'Women and Child']
    import ast
    parsed = ast.literal_eval(str(value))

    if not isinstance(parsed, list):
        raise ValueError(f"Expected a category list, got: {value}")

    return [str(item) for item in parsed]

df_step5["category_list"] = df_step5["category_list"].apply(
    parse_category_list
)


In [143]:
# 3. Check scheme identity and index-mapping prerequisites
print("=" * 55)
print("IDENTITY AND MAPPING CHECKS")
print("=" * 55)

print("Dataset rows:", len(df_step5))
print("Unique scheme IDs:", df_step5["scheme_id"].nunique())
print("Duplicate scheme IDs:", int(df_step5["scheme_id"].duplicated().sum()))
print("Duplicate source row IDs:", int(df_step5["source_row_id"].duplicated().sum()))
print("Empty scheme names:", int(df_step5["scheme_name"].fillna("").str.strip().eq("").sum()))
print("Empty retrieval text:", int(df_step5["retrieval_text"].fillna("").str.strip().eq("").sum()))

assert df_step5["scheme_id"].is_unique, "scheme_id values must be unique."
assert df_step5["source_row_id"].is_unique, "source_row_id values must be unique."
assert df_step5["retrieval_text"].fillna("").str.strip().ne("").all(), \
    "Some schemes have empty retrieval text."

IDENTITY AND MAPPING CHECKS
Dataset rows: 3397
Unique scheme IDs: 3397
Duplicate scheme IDs: 0
Duplicate source row IDs: 0
Empty scheme names: 0
Empty retrieval text: 0


In [144]:
# 4. Inspect repeated retrieval text
duplicate_text_mask = df_step5["retrieval_text"].duplicated(keep=False)

print("\nEXACT RETRIEVAL TEXT DUPLICATES")
print("Rows in duplicate-text groups:", int(duplicate_text_mask.sum()))

if duplicate_text_mask.any():
    display(
        df_step5.loc[
            duplicate_text_mask,
            ["scheme_id", "scheme_name", "slug"]
        ].sort_values("scheme_name").head(20)
    )


EXACT RETRIEVAL TEXT DUPLICATES
Rows in duplicate-text groups: 0


In [145]:
# 5. Inspect completeness of useful answer fields
answer_fields = [
    "details",
    "benefits",
    "eligibility",
    "application",
    "documents",
    "level",
    "tags",
]

print("\nFIELD COMPLETENESS")
completeness_rows = []

for col in answer_fields:
    values = df_step5[col].fillna("").astype(str).str.strip()
    completeness_rows.append({
        "field": col,
        "empty_count": int(values.eq("").sum()),
        "filled_count": int(values.ne("").sum()),
        "filled_percent": round(float(values.ne("").mean() * 100), 2),
    })

display(pd.DataFrame(completeness_rows))


FIELD COMPLETENESS


,field,empty_count,filled_count,filled_percent
0,details,0,3397,100.00
1,benefits,0,3397,100.00
2,eligibility,0,3397,100.00
3,application,2,3395,99.94
4,documents,11,3386,99.68
5,level,0,3397,100.00
6,tags,29,3368,99.15


In [146]:
# 6. Inspect retrieval-text lengths
text_lengths = df_step5["retrieval_text"].fillna("").str.len()

print("\nRETRIEVAL TEXT LENGTHS")
print("Minimum characters:", int(text_lengths.min()))
print("Median characters:", int(text_lengths.median()))
print("95th percentile:", int(text_lengths.quantile(0.95)))
print("Maximum characters:", int(text_lengths.max()))

print("\nLongest records (review for unusually large text)")
display(
    df_step5.assign(retrieval_text_length=text_lengths)
    .nlargest(5, "retrieval_text_length")[
        ["scheme_id", "scheme_name", "retrieval_text_length"]
    ]
)



RETRIEVAL TEXT LENGTHS
Minimum characters: 934
Median characters: 3301
95th percentile: 7559
Maximum characters: 24981

Longest records (review for unusually large text)


,scheme_id,scheme_name,retrieval_text_length
947,SCHEME_facapabgist,Financial Assistance For Cultural Activities I...,24981
2414,SCHEME_pmfmpe,Pradhan Mantri Formalisation Of Micro Food Pro...,23063
1680,SCHEME_ky-smsp,Krishonnati Yojana - Sub Mission On Seed And P...,20969
2152,SCHEME_nfandshests,National Fellowship & Scholarship For Higher E...,20635
946,SCHEME_faaca,Financial Assistance For Allied Cultural Activ...,19469


In [147]:
# 7. Prepare structured metadata for retrieval.
# Keep original source fields for factual answers.
metadata_columns = [
    "source_row_id",
    "scheme_id",
    "scheme_name",
    "slug",
    "level",
    "schemeCategory",
    "normalized_categories",
    "category_list",
    "eligibility",
    "benefits",
    "application",
    "documents",
    "details",
]

metadata = df_step5[metadata_columns].copy()

# JSON-encode categories for consistent downstream parsing.
metadata["category_list"] = metadata["category_list"].apply(
    lambda values: json.dumps(values, ensure_ascii=False)
)

In [148]:
# 8. Save the retrieval-ready dataset and metadata separately.
df_step5.to_csv("retrieval_dataset_v4.csv", index=False)
metadata.to_csv("scheme_metadata_v4.csv", index=False)

print("\nSAVED FILES")
print("retrieval_dataset_v4.csv")
print("scheme_metadata_v4.csv")
print("\nStep 5 completed.")



SAVED FILES
retrieval_dataset_v4.csv
scheme_metadata_v4.csv

Step 5 completed.


In [149]:
%pip install -q sentence-transformers faiss-cpu

In [150]:
# Install once in Google Colab if needed:
# %pip install -q sentence-transformers faiss-cpu

import os
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [151]:
# --------------------------------------------------
# 1. Load the validated dataset
# --------------------------------------------------
DATA_PATH = "retrieval_dataset_v4.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

required = [
    "scheme_id", "scheme_name", "level",
    "normalized_categories", "details", "benefits",
    "eligibility", "application", "documents"
]

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if df["scheme_id"].isna().any() or not df["scheme_id"].is_unique:
    raise ValueError("scheme_id must be present and unique.")


In [152]:
# --------------------------------------------------
# 2. Split long fields into overlapping word chunks
# --------------------------------------------------
CHUNK_WORDS = 160
OVERLAP_WORDS = 30

def split_into_chunks(text, chunk_words=CHUNK_WORDS,
                      overlap_words=OVERLAP_WORDS):
    text = str(text).strip()

    if not text:
        return []

    words = text.split()

    if len(words) <= chunk_words:
        return [text]

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_words, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap_words

    return chunks

# Search-focused fields. Complete source fields remain in df.
search_fields = [
    ("description", "details"),
    ("benefits", "benefits"),
    ("eligibility", "eligibility"),
    ("application", "application"),
    ("documents", "documents"),
]

chunk_records = []

for _, row in df.iterrows():
    scheme_id = str(row["scheme_id"])
    scheme_name = str(row["scheme_name"]).strip()
    level = str(row["level"]).strip()
    categories = str(row["normalized_categories"]).strip()

    for field_type, column in search_fields:
        value = row[column]

        if pd.isna(value) or not str(value).strip():
            continue

        pieces = split_into_chunks(value)

        for chunk_number, piece in enumerate(pieces):
            # Repeat concise scheme context in each chunk.
            chunk_text = (
                f"Scheme: {scheme_name}\n"
                f"Level: {level}\n"
                f"Categories: {categories}\n"
                f"Information type: {field_type}\n"
                f"{piece}"
            )

            chunk_records.append({
                "chunk_id": len(chunk_records),
                "scheme_id": scheme_id,
                "scheme_name": scheme_name,
                "field_type": field_type,
                "chunk_number": chunk_number,
                "chunk_text": chunk_text,
            })

chunks_df = pd.DataFrame(chunk_records)

if chunks_df.empty:
    raise ValueError("No text chunks were generated.")

if chunks_df["chunk_text"].str.strip().eq("").any():
    raise ValueError("Empty chunks were generated.")

print("Schemes:", len(df))
print("Search chunks:", len(chunks_df))
print("Unique mapped schemes:", chunks_df["scheme_id"].nunique())
print("\nChunks by field:")
print(chunks_df["field_type"].value_counts().to_string())

Schemes: 3397
Search chunks: 22068
Unique mapped schemes: 3397

Chunks by field:
field_type
application    5663
description    4363
eligibility    4186
benefits       4081
documents      3775


In [153]:
# --------------------------------------------------
# 3. Load the same embedding model used at query time
# --------------------------------------------------
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
encoder = SentenceTransformer(MODEL_NAME)

# SentenceTransformer may truncate inputs exceeding its
# maximum token length. Chunking limits that risk.
print("\nEncoder maximum sequence length:", encoder.max_seq_length)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Encoder maximum sequence length: 256


In [154]:
# --------------------------------------------------
# 4. Generate normalized embeddings
# --------------------------------------------------
texts = chunks_df["chunk_text"].tolist()

embeddings = encoder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

if not np.isfinite(embeddings).all():
    raise ValueError("Embeddings contain NaN or infinite values.")


Batches:   0%|          | 0/690 [00:00<?, ?it/s]

In [155]:
# --------------------------------------------------
# 5. Build cosine-similarity FAISS index
# --------------------------------------------------
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [156]:
# --------------------------------------------------
# 6. Save chunks, embeddings, index and model identifier
# --------------------------------------------------
chunks_path = os.path.join(MODEL_DIR, "scheme_chunks.csv")
embeddings_path = os.path.join(MODEL_DIR, "scheme_embeddings.npy")
index_path = os.path.join(MODEL_DIR, "scheme_index.faiss")
encoder_path = os.path.join(MODEL_DIR, "scheme_encoder_name.txt")

chunks_df.to_csv(chunks_path, index=False)
np.save(embeddings_path, embeddings)
faiss.write_index(index, index_path)

with open(encoder_path, "w", encoding="utf-8") as f:
    f.write(MODEL_NAME)


In [157]:
# --------------------------------------------------
# 7. Validate row order and index alignment
# --------------------------------------------------
saved_chunks = pd.read_csv(chunks_path)
saved_embeddings = np.load(embeddings_path)
saved_index = faiss.read_index(index_path)

assert len(saved_chunks) == len(chunks_df)
assert saved_embeddings.shape[0] == len(saved_chunks)
assert saved_index.ntotal == len(saved_chunks)
assert saved_embeddings.shape[1] == saved_index.d

print("\nSTEP 6 VALIDATION")
print("Dataset schemes:", len(df))
print("Chunk rows:", len(saved_chunks))
print("Embedding shape:", saved_embeddings.shape)
print("FAISS vectors:", saved_index.ntotal)
print("FAISS dimension:", saved_index.d)
print("Index metric: inner product (normalized vectors = cosine similarity)")
print("Unique scheme IDs in chunks:", saved_chunks["scheme_id"].nunique())

# Check a few stored vectors against the matching saved embeddings.
sample_ids = sorted(set([
    0,
    len(saved_chunks) // 2,
    len(saved_chunks) - 1
]))

for i in sample_ids:
    stored = saved_index.reconstruct(i)
    matches = np.allclose(
        stored, saved_embeddings[i], rtol=1e-5, atol=1e-6
    )
    print(f"Vector {i} alignment: {matches}")
    assert matches, f"Vector mismatch at FAISS ID {i}"

print("\nStep 6 completed successfully.")



STEP 6 VALIDATION
Dataset schemes: 3397
Chunk rows: 22068
Embedding shape: (22068, 384)
FAISS vectors: 22068
FAISS dimension: 384
Index metric: inner product (normalized vectors = cosine similarity)
Unique scheme IDs in chunks: 3397
Vector 0 alignment: True
Vector 11034 alignment: True
Vector 22067 alignment: True

Step 6 completed successfully.


In [158]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [159]:
# --------------------------------------------------
# 1. Load the artifacts created in Step 6
# --------------------------------------------------
MODEL_DIR = "model"

chunks_df = pd.read_csv(f"{MODEL_DIR}/scheme_chunks.csv")
index = faiss.read_index(f"{MODEL_DIR}/scheme_index.faiss")
encoder_name = open(
    f"{MODEL_DIR}/scheme_encoder_name.txt",
    encoding="utf-8"
).read().strip()

encoder = SentenceTransformer(encoder_name)

# Verify index-to-chunk alignment
assert len(chunks_df) == index.ntotal
assert chunks_df["scheme_id"].notna().all()
assert chunks_df["chunk_text"].notna().all()

print("Chunks loaded:", len(chunks_df))
print("FAISS vectors:", index.ntotal)
print("Encoder:", encoder_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks loaded: 22068
FAISS vectors: 22068
Encoder: sentence-transformers/all-MiniLM-L6-v2


In [160]:
# --------------------------------------------------
# 2. Semantic retrieval function
# --------------------------------------------------
def retrieve_schemes(question, top_k_chunks=10, top_k_schemes=5):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please enter a non-empty question.")

    # Normalize query embeddings exactly as in Step 6
    query_embedding = encoder.encode(
        [question.strip()],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve candidate chunks
    scores, ids = index.search(
        query_embedding,
        min(top_k_chunks, index.ntotal)
    )

    results = []

    for score, chunk_id in zip(scores[0], ids[0]):
        if chunk_id < 0:
            continue

        row = chunks_df.iloc[int(chunk_id)]

        results.append({
            "faiss_id": int(chunk_id),
            "similarity": float(score),
            "scheme_id": row["scheme_id"],
            "scheme_name": row["scheme_name"],
            "field_type": row["field_type"],
            "chunk_number": int(row["chunk_number"]),
            "matched_text": row["chunk_text"]
        })

    if not results:
        return pd.DataFrame()

    # Keep the strongest matching chunk for each scheme.
    # This avoids showing the same scheme repeatedly.
    results_df = pd.DataFrame(results)

    results_df = (
        results_df.sort_values("similarity", ascending=False)
        .drop_duplicates(subset="scheme_id", keep="first")
        .head(top_k_schemes)
        .reset_index(drop=True)
    )

    return results_df

In [161]:
# --------------------------------------------------
# 3. Try one question first
# --------------------------------------------------
question = "What financial assistance is available for farmers?"

results = retrieve_schemes(
    question,
    top_k_chunks=20,
    top_k_schemes=5
)

display(
    results[
        [
            "faiss_id",
            "similarity",
            "scheme_id",
            "scheme_name",
            "field_type"
        ]
    ]
)

# Show the text that caused the match
for i, row in results.iterrows():
    print(f"\n{'=' * 70}")
    print(f"Rank: {i + 1}")
    print("Scheme:", row["scheme_name"])
    print("Scheme ID:", row["scheme_id"])
    print("Matched field:", row["field_type"])
    print("Similarity:", round(row["similarity"], 4))
    print("Matched text:")
    print(row["matched_text"][:1000])


,faiss_id,similarity,scheme_id,scheme_name,field_type
0,662,0.672176,SCHEME_agrasp,AGR-61 Agro Service Providers for the Purpose ...,benefits
1,1300,0.672147,SCHEME_aapd,Agricultural Activities for Persons with Disab...,benefits
2,6206,0.650988,SCHEME_faspalscf,Financial Assistance Scheme for Purchase of Ag...,benefits
3,15385,0.637032,SCHEME_pcds,Plantation Crop Development Scheme,benefits
4,6731,0.635213,SCHEME_fabwaf,Financial Assistance to Brackish Water Aquacul...,benefits



Rank: 1
Scheme: AGR-61 Agro Service Providers for the Purpose of Making Farming More Profitable With the Help of Farm Mechanization
Scheme ID: SCHEME_agrasp
Matched field: benefits
Similarity: 0.6722
Matched text:
Scheme: AGR-61 Agro Service Providers for the Purpose of Making Farming More Profitable With the Help of Farm Mechanization
Level: State
Categories: Agriculture,Rural & Environment
Information type: benefits
﻿ Name of the Component Standard of Assistance (General areas) Standard of Assistance (Tribal areas) Agro Service Provider Unit Up to 75% or limited to ₹7,50,000/- (40% in the 1st year, 10% in the 2nd and 3rd years, and 7.5% in the 4th and 5th years) Up to 85% or limited to ₹8,50,000/- (50% in the 1st year, 10% in the 2nd and 3rd years, and 7.5% in the 4th and 5th years) Note: Assistance is available once in a lifetime for eligible applicants.

Rank: 2
Scheme: Agricultural Activities for Persons with Disabilities
Scheme ID: SCHEME_aapd
Matched field: benefits
Similarity:

In [162]:
test_questions = [
    "What financial assistance is available for farmers?",
    "Which scholarships are available for college students?",
    "What government benefits are available for women?",
    "What documents are required to apply for a scheme?",
    "How can an unemployed person get skill training?",
    "Are there schemes for senior citizens?",
    "What support is available for small businesses?",
    "Which schemes are available in Puducherry?"
]

for question in test_questions:
    print("\n" + "=" * 85)
    print("QUESTION:", question)

    results = retrieve_schemes(
        question,
        top_k_chunks=20,
        top_k_schemes=5
    )

    if results.empty:
        print("No results returned.")
        continue

    display(
        results[
            [
                "similarity",
                "scheme_name",
                "field_type",
                "scheme_id"
            ]
        ]
    )



QUESTION: What financial assistance is available for farmers?


,similarity,scheme_name,field_type,scheme_id
0,0.672176,AGR-61 Agro Service Providers for the Purpose ...,benefits,SCHEME_agrasp
1,0.672147,Agricultural Activities for Persons with Disab...,benefits,SCHEME_aapd
2,0.650988,Financial Assistance Scheme for Purchase of Ag...,benefits,SCHEME_faspalscf
3,0.637032,Plantation Crop Development Scheme,benefits,SCHEME_pcds
4,0.635213,Financial Assistance to Brackish Water Aquacul...,benefits,SCHEME_fabwaf



QUESTION: Which scholarships are available for college students?


,similarity,scheme_name,field_type,scheme_id
0,0.583159,Scholarships For Higher Education Abroad To Me...,eligibility,SCHEME_sheambgoc
1,0.575535,Scheme for Award of Scholarships to School or ...,eligibility,SCHEME_sasseceie
2,0.561615,IASRI Scholarship For M.Sc And Ph.D,benefits,SCHEME_iasri-scholarship
3,0.556564,Pre Matric Scholarship For Minorities,application,SCHEME_presm
4,0.549104,Post-Graduate Scholarships for Professional Co...,documents,SCHEME_pgspcscstc



QUESTION: What government benefits are available for women?


,similarity,scheme_name,field_type,scheme_id
0,0.573204,Pension To Widows And Destitute Women-Chandigarh,description,SCHEME_ptwadwc
1,0.556670,Women Helpline,benefits,SCHEME_whl-181
2,0.545724,Maternity Benefit,benefits,SCHEME_mbwb
3,0.536299,Incentive to Widow's Remarriage,benefits,SCHEME_iwr
4,0.531277,Maternity Benefit to the Female Beneficiary,benefits,SCHEME_mbttfb


... (6 outputs left out)


QUESTION: What support is available for small businesses?


,similarity,scheme_name,field_type,scheme_id
0,0.533502,Entrepreneur Support Scheme,description,SCHEME_ess
1,0.486115,Subsidy for Developing Business Premises for P...,benefits,SCHEME_sdbppd
2,0.480572,Small Industries Unit for Persons with Disabil...,benefits,SCHEME_siupd
3,0.479642,"PRISM (Promoting Innovations in Individuals, S...",benefits,SCHEME_prism-c1-pcpm
4,0.456942,"Scheme for Assistance to Micro, Small and Medi...",eligibility,SCHEME_samsmeisss



QUESTION: Which schemes are available in Puducherry?


,similarity,scheme_name,field_type,scheme_id
0,0.566295,'Organizing/ Participation in Exhibitions /Tra...,description,SCHEME_opetfesime-mp
1,0.559998,"""Exhibition"" Component of the ""Development Of ...",eligibility,SCHEME_ecdhs
2,0.549458,"""Advance / High Skill Training"" Component of t...",description,SCHEME_ahst
3,0.544363,"""Market Development Support Assistance"" under ...",description,SCHEME_mdsa-mesifai-vi
4,0.542865,Financial Assistance to Indigent Artistes,description,SCHEME_fatia


In [163]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load saved artifacts
chunks_df = pd.read_csv("model/scheme_chunks.csv")
index = faiss.read_index("model/scheme_index.faiss")

with open("model/scheme_encoder_name.txt", encoding="utf-8") as f:
    encoder_name = f.read().strip()

encoder = SentenceTransformer(encoder_name)

assert len(chunks_df) == index.ntotal
assert chunks_df["scheme_id"].notna().all()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [164]:
# Query types are inferred with simple rules for this baseline.
# We will refine these rules based on the test results.
FIELD_KEYWORDS = {
    "eligibility": [
        "eligible", "eligibility", "qualify", "qualification",
        "who can apply", "age limit", "income limit"
    ],
    "benefits": [
        "benefit", "benefits", "financial assistance", "subsidy",
        "loan", "amount", "how much", "support"
    ],
    "documents": [
        "document", "documents", "proof", "certificate", "required papers"
    ],
    "application": [
        "apply", "application", "how to apply", "register",
        "registration", "where to apply"
    ],
    "description": [
        "what is", "overview", "purpose", "about this scheme"
    ]
}

def infer_requested_field(question):
    q = question.casefold()

    matches = []
    for field, phrases in FIELD_KEYWORDS.items():
        if any(phrase in q for phrase in phrases):
            matches.append(field)

    return matches

def retrieve_ranked(question, top_k_chunks=50, top_k_schemes=5):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    qvec = encoder.encode(
        [question.strip()],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(
        qvec, min(top_k_chunks, index.ntotal)
    )

    requested_fields = infer_requested_field(question)
    results = []

    for score, chunk_id in zip(scores[0], ids[0]):
        if chunk_id < 0:
            continue

        row = chunks_df.iloc[int(chunk_id)]
        field_type = str(row["field_type"])

        # A modest boost for chunks whose field matches the query.
        # Similarity remains the primary ranking signal.
        field_boost = (
            0.04 if field_type in requested_fields else 0.0
        )

        results.append({
            "faiss_id": int(chunk_id),
            "similarity": float(score),
            "field_boost": field_boost,
            "ranking_score": float(score) + field_boost,
            "scheme_id": row["scheme_id"],
            "scheme_name": row["scheme_name"],
            "field_type": field_type,
            "chunk_text": row["chunk_text"]
        })

    if not results:
        return pd.DataFrame()

    result_df = pd.DataFrame(results)

    # First rank matching chunks, then keep the best chunk per scheme.
    result_df = (
        result_df.sort_values("ranking_score", ascending=False)
        .drop_duplicates("scheme_id", keep="first")
        .head(top_k_schemes)
        .reset_index(drop=True)
    )

    return result_df

In [165]:
# Run the same questions so we can compare rankings.
test_questions = [
    "What financial assistance is available for farmers?",
    "Which scholarships are available for college students?",
    "What government benefits are available for women?",
    "What documents are required to apply for a scheme?",
    "How can an unemployed person get skill training?",
    "Are there schemes for senior citizens?",
    "What support is available for small businesses?",
    "Which schemes are available in Puducherry?"
]

for question in test_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("Requested fields:", infer_requested_field(question))

    results = retrieve_ranked(question)

    if results.empty:
        print("No results.")
        continue

    display(results[
        [
            "faiss_id",
            "similarity",
            "field_boost",
            "ranking_score",
            "scheme_id",
            "scheme_name",
            "field_type"
        ]
    ])



QUESTION: What financial assistance is available for farmers?
Requested fields: ['benefits']


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,field_type
0,662,0.672176,0.04,0.712176,SCHEME_agrasp,AGR-61 Agro Service Providers for the Purpose ...,benefits
1,1300,0.672147,0.04,0.712147,SCHEME_aapd,Agricultural Activities for Persons with Disab...,benefits
2,6206,0.650988,0.04,0.690988,SCHEME_faspalscf,Financial Assistance Scheme for Purchase of Ag...,benefits
3,15385,0.637032,0.04,0.677032,SCHEME_pcds,Plantation Crop Development Scheme,benefits
4,6731,0.635213,0.04,0.675213,SCHEME_fabwaf,Financial Assistance to Brackish Water Aquacul...,benefits



QUESTION: Which scholarships are available for college students?
Requested fields: []


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,field_type
0,18838,0.583159,0.0,0.583159,SCHEME_sheambgoc,Scholarships For Higher Education Abroad To Me...,eligibility
1,17961,0.575535,0.0,0.575535,SCHEME_sasseceie,Scheme for Award of Scholarships to School or ...,eligibility
2,8688,0.561615,0.0,0.561615,SCHEME_iasri-scholarship,IASRI Scholarship For M.Sc And Ph.D,benefits
3,15989,0.556564,0.0,0.556564,SCHEME_presm,Pre Matric Scholarship For Minorities,application
4,15613,0.549104,0.0,0.549104,SCHEME_pgspcscstc,Post-Graduate Scholarships for Professional Co...,documents



QUESTION: What government benefits are available for women?
Requested fields: ['benefits']


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,field_type
0,21808,0.556670,0.04,0.596670,SCHEME_whl-181,Women Helpline,benefits
1,11887,0.545724,0.04,0.585724,SCHEME_mbwb,Maternity Benefit,benefits
2,9081,0.536299,0.04,0.576299,SCHEME_iwr,Incentive to Widow's Remarriage,benefits
3,15219,0.573204,0.00,0.573204,SCHEME_ptwadwc,Pension To Widows And Destitute Women-Chandigarh,description
4,11984,0.531277,0.04,0.571277,SCHEME_mbttfb,Maternity Benefit to the Female Beneficiary,benefits


... (6 outputs left out)


QUESTION: What support is available for small businesses?
Requested fields: ['benefits']


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,field_type
0,5703,0.533502,0.00,0.533502,SCHEME_ess,Entrepreneur Support Scheme,description
1,20089,0.486115,0.04,0.526115,SCHEME_sdbppd,Subsidy for Developing Business Premises for P...,benefits
2,19424,0.480572,0.04,0.520572,SCHEME_siupd,Small Industries Unit for Persons with Disabil...,benefits
3,14868,0.479642,0.04,0.519642,SCHEME_prism-c1-pcpm,"PRISM (Promoting Innovations in Individuals, S...",benefits
4,17922,0.456598,0.04,0.496598,SCHEME_samsmesme,"Scheme for Assistance to Micro, Small and Medi...",benefits



QUESTION: Which schemes are available in Puducherry?
Requested fields: []


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,field_type
0,575,0.566295,0.0,0.566295,SCHEME_opetfesime-mp,'Organizing/ Participation in Exhibitions /Tra...,description
1,331,0.559998,0.0,0.559998,SCHEME_ecdhs,"""Exhibition"" Component of the ""Development Of ...",eligibility
2,207,0.549458,0.0,0.549458,SCHEME_ahst,"""Advance / High Skill Training"" Component of t...",description
3,456,0.544363,0.0,0.544363,SCHEME_mdsa-mesifai-vi,"""Market Development Support Assistance"" under ...",description
4,6763,0.542865,0.0,0.542865,SCHEME_fatia,Financial Assistance to Indigent Artistes,description


In [166]:
import pandas as pd
import ast

# Load the processed dataset and metadata
schemes = pd.read_csv("processed_schemes_v3.csv")
metadata = pd.read_csv("scheme_metadata_v4.csv")

print("SCHEMES COLUMNS")
print(schemes.columns.tolist())

print("\nMETADATA COLUMNS")
print(metadata.columns.tolist())


SCHEMES COLUMNS
['source_row_id', 'scheme_name', 'slug', 'details', 'benefits', 'eligibility', 'application', 'documents', 'level', 'schemeCategory', 'tags', 'scheme_id', 'retrieval_text', 'category_list', 'category_unrecognized', 'normalized_categories']

METADATA COLUMNS
['source_row_id', 'scheme_id', 'scheme_name', 'slug', 'level', 'schemeCategory', 'normalized_categories', 'category_list', 'eligibility', 'benefits', 'application', 'documents', 'details']


In [167]:
# Inspect geography-related fields
geo_candidates = [
    col for col in schemes.columns
    if any(term in col.lower() for term in [
        "state", "location", "region", "district", "level",
        "geograph", "jurisdiction"
    ])
]

print("\nGEOGRAPHY-RELATED COLUMNS")
print(geo_candidates)

for col in geo_candidates:
    print(f"\n--- {col}: top 25 values ---")
    print(schemes[col].fillna("(missing)").value_counts().head(25))

# Inspect scheme levels and category representations
if "level" in schemes.columns:
    print("\nSCHEME LEVELS")
    print(schemes["level"].fillna("(missing)").value_counts().head(30))

if "category_list" in schemes.columns:
    print("\nCATEGORY LIST EXAMPLES")
    print(schemes["category_list"].head(10).tolist())


GEOGRAPHY-RELATED COLUMNS
['level']

--- level: top 25 values ---
level
State      2856
Central     541
Name: count, dtype: int64

SCHEME LEVELS
level
State      2856
Central     541
Name: count, dtype: int64

CATEGORY LIST EXAMPLES
["['Agriculture,Rural & Environment', 'Social welfare & Empowerment']", "['Education & Learning']", "['Social welfare & Empowerment']", "['Business & Entrepreneurship']", "['Social welfare & Empowerment']", "['Social welfare & Empowerment']", "['Business & Entrepreneurship']", "['Women and Child']", "['Social welfare & Empowerment']", "['Health & Wellness']"]


In [168]:
# Check whether the chunks preserve the metadata needed for filtering
chunks = pd.read_csv("model/scheme_chunks.csv")

print("\nCHUNK COLUMNS")
print(chunks.columns.tolist())

for col in ["scheme_id", "scheme_name", "level", "categories", "field_type"]:
    if col in chunks.columns:
        print(f"\n{col}:")
        print(chunks[col].fillna("(missing)").value_counts().head(15))


CHUNK COLUMNS
['chunk_id', 'scheme_id', 'scheme_name', 'field_type', 'chunk_number', 'chunk_text']

scheme_id:
scheme_id
SCHEME_facapabgist                    31
SCHEME_pmfmpe                         28
SCHEME_ky-smsp                        28
SCHEME_nfandshests                    25
SCHEME_faaca                          25
SCHEME_goc                            24
SCHEME_financial-assistance-scheme    23
SCHEME_tnfcr                          22
SCHEME_ess                            21
SCHEME_kuy-movcdner                   21
SCHEME_sadppfaa                       20
SCHEME_tnscr                          20
SCHEME_css-nos-st                     18
SCHEME_ns-icarif                      18
SCHEME_sgiavowwst                     18
Name: count, dtype: int64

scheme_name:
scheme_name
Financial Assistance For Cultural Activities In Performing Arts For Building Grants Including Studio Theatres                  31
Pradhan Mantri Formalisation Of Micro Food Processing Enterprises                

In [169]:

import pandas as pd
import ast

# Load the existing data
chunks = pd.read_csv("model/scheme_chunks.csv")
schemes = pd.read_csv("processed_schemes_v3.csv")

In [170]:
# Check the join key
assert chunks["scheme_id"].notna().all(), "Missing scheme IDs in chunks"
assert schemes["scheme_id"].is_unique, "Scheme IDs are not unique"

# Keep the metadata needed for filtering
metadata_cols = [
    "scheme_id",
    "level",
    "normalized_categories",
    "category_list",
    "schemeCategory"
]

metadata = schemes[metadata_cols].copy()

In [171]:
# Join scheme metadata onto every chunk
chunks_enriched = chunks.merge(
    metadata,
    on="scheme_id",
    how="left",
    validate="many_to_one",
    indicator=True
)


In [172]:

# Verify that every chunk matched a scheme
print("Chunk rows:", len(chunks_enriched))
print("Unmatched chunks:", (chunks_enriched["_merge"] != "both").sum())

chunks_enriched.drop(columns="_merge", inplace=True)

# Convert serialized category lists into Python lists
def parse_list(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

chunks_enriched["category_list_parsed"] = (
    chunks_enriched["category_list"].apply(parse_list)
)


Chunk rows: 22068
Unmatched chunks: 0


In [173]:
# Inspect the enriched metadata
print("\nColumns:")
print(chunks_enriched.columns.tolist())

print("\nScheme levels:")
print(chunks_enriched["level"].value_counts(dropna=False))

print("\nExample enriched chunks:")
print(
    chunks_enriched[
        ["scheme_name", "level", "category_list_parsed", "field_type"]
    ].head(5).to_string(index=False)
)


Columns:
['chunk_id', 'scheme_id', 'scheme_name', 'field_type', 'chunk_number', 'chunk_text', 'level', 'normalized_categories', 'category_list', 'schemeCategory', 'category_list_parsed']

Scheme levels:
level
State      17446
Central     4622
Name: count, dtype: int64

Example enriched chunks:
                                                                                                             scheme_name level                                            category_list_parsed  field_type
"Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" State [Agriculture,Rural & Environment, Social welfare & Empowerment] description
"Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" State [Agriculture,Rural & Environment, Social welfare & Empowerment]    benefits
"Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and N

In [174]:
# Save a separate file so the original chunks remain unchanged
chunks_enriched.to_csv(
    "model/scheme_chunks_enriched.csv",
    index=False
)

print("\nSaved: model/scheme_chunks_enriched.csv")



Saved: model/scheme_chunks_enriched.csv


In [175]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load existing retrieval artifacts
chunks_df = pd.read_csv("model/scheme_chunks_enriched.csv")
index = faiss.read_index("model/scheme_index.faiss")

with open("model/scheme_encoder_name.txt", "r") as f:
    encoder_name = f.read().strip()

encoder = SentenceTransformer(encoder_name)

assert len(chunks_df) == index.ntotal, (
    f"Chunk/index mismatch: {len(chunks_df)} chunks vs {index.ntotal} vectors"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [176]:
# Questions that explicitly request information from a particular field
FIELD_KEYWORDS = {
    "eligibility": [
        "eligibility", "eligible", "qualify", "qualification",
        "who can apply", "age limit", "income limit"
    ],
    "benefits": [
        "benefit", "benefits", "financial assistance", "subsidy",
        "loan", "amount", "how much", "support"
    ],
    "documents": [
        "document", "documents", "proof", "certificate",
        "required papers"
    ],
    "application": [
        "apply", "application", "how to apply",
        "register", "registration", "where to apply"
    ],
    "description": [
        "what is", "overview", "purpose", "about this scheme"
    ]
}


def infer_requested_fields(question):
    q = question.lower()
    fields = []

    for field, keywords in FIELD_KEYWORDS.items():
        if any(keyword in q for keyword in keywords):
            fields.append(field)

    return fields


def retrieve_schemes(
    question,
    top_k_chunks=100,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    field_boost=0.04
):
    """
    level_filter:
        None, "State", or "Central"

    category_filter:
        None or a canonical category name, e.g.
        "Education & Learning"
    """

    query_vector = encoder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    search_k = min(top_k_chunks, index.ntotal)
    similarities, faiss_ids = index.search(query_vector, search_k)

    candidates = chunks_df.iloc[faiss_ids[0]].copy()
    candidates["faiss_id"] = faiss_ids[0]
    candidates["similarity"] = similarities[0]

    # Ignore invalid FAISS results, if any
    candidates = candidates[candidates["faiss_id"] >= 0].copy()

    # Optional scheme-level filter
    if level_filter is not None:
        candidates = candidates[
            candidates["level"].str.casefold()
            == level_filter.casefold()
        ]

    # Optional category filter
    if category_filter is not None:
        category_mask = candidates["category_list_parsed"].apply(
            lambda categories: (
                category_filter.casefold()
                in [str(c).casefold() for c in categories]
            )
        )
        candidates = candidates[category_mask]

    if candidates.empty:
        print("No matching schemes found for these filters.")
        return pd.DataFrame()

    # Boost chunks matching the requested information field
    requested_fields = infer_requested_fields(question)

    candidates["field_boost"] = candidates["field_type"].apply(
        lambda field: (
            field_boost if field in requested_fields else 0.0
        )
    )

    candidates["ranking_score"] = (
        candidates["similarity"] + candidates["field_boost"]
    )

    # Rank chunks, then keep the strongest chunk for each scheme
    candidates = candidates.sort_values(
        "ranking_score", ascending=False
    )

    best_per_scheme = candidates.drop_duplicates(
        subset=["scheme_id"], keep="first"
    )

    result_cols = [
        "faiss_id", "similarity", "field_boost", "ranking_score",
        "scheme_id", "scheme_name", "level", "category_list_parsed",
        "field_type"
    ]

    return best_per_scheme[result_cols].head(top_k_schemes).reset_index(
        drop=True
    )



In [177]:
# Test 1: General semantic search
print("TEST 1: Farmer financial assistance")
print(
    retrieve_schemes(
        "What financial assistance is available for farmers?"
    ).to_string(index=False)
)


TEST 1: Farmer financial assistance
 faiss_id  similarity  field_boost  ranking_score        scheme_id                                                                                                         scheme_name level                                                         category_list_parsed field_type
      662    0.672176         0.04       0.712176    SCHEME_agrasp AGR-61 Agro Service Providers for the Purpose of Making Farming More Profitable With the Help of Farm Mechanization State                                          ['Agriculture,Rural & Environment']   benefits
     1300    0.672147         0.04       0.712147      SCHEME_aapd                                                               Agricultural Activities for Persons with Disabilities State           ['Agriculture,Rural & Environment', 'Business & Entrepreneurship']   benefits
     6206    0.650988         0.04       0.690988 SCHEME_faspalscf                            Financial Assistance Scheme for Purchas

In [178]:

# Test 2: Category filtering
print("\nTEST 2: Education category")
print(
    retrieve_schemes(
        "Which scholarships are available for college students?",
        category_filter="Education & Learning"
    ).to_string(index=False)
)


TEST 2: Education category
No matching schemes found for these filters.
Empty DataFrame
Columns: []
Index: []


In [179]:

# Test 3: Central schemes only
print("\nTEST 3: Central schemes")
print(
    retrieve_schemes(
        "What financial support is available for small businesses?",
        level_filter="Central"
    ).to_string(index=False)
)



TEST 3: Central schemes
 faiss_id  similarity  field_boost  ranking_score                      scheme_id                                                                                                                                                                                             scheme_name   level                                           category_list_parsed field_type
    20568    0.476667         0.04       0.516667                    SCHEME_tdfs                                                                                                                                                                Technology Development Fund (TDF) Scheme Central                                ['Business & Entrepreneurship']   benefits
    14868    0.463355         0.04       0.503355           SCHEME_prism-c1-pcpm                                                                                     PRISM (Promoting Innovations in Individuals, Start-ups and MSMEs): Category-1: Proof

In [180]:

# Test 4: Documents question
print("\nTEST 4: Documents")
print(
    retrieve_schemes(
        "What documents are required to apply for a scheme?"
    ).to_string(index=False)
)



TEST 4: Documents
 faiss_id  similarity  field_boost  ranking_score    scheme_id                         scheme_name   level                         category_list_parsed field_type
     7096    0.731988         0.04       0.771988  SCHEME_fcds     Floriculture Development Scheme   State          ['Agriculture,Rural & Environment']  documents
     7322    0.726465         0.04       0.766465   SCHEME_fds            Fruit Development Scheme   State          ['Agriculture,Rural & Environment']  documents
     1371    0.720244         0.04       0.760244  SCHEME_aius Agriculture Information Unit Scheme   State          ['Agriculture,Rural & Environment']  documents
    15600    0.716756         0.04       0.756756 SCHEME_pomis   Post Office Monthly Income Scheme Central ['Banking,Financial Services and Insurance']  documents
    15378    0.716699         0.04       0.756699   SCHEME_pps             Plant Protection Scheme   State          ['Agriculture,Rural & Environment']  documents


In [181]:
import pandas as pd
import numpy as np
import ast

# Reload enriched chunks
chunks_df = pd.read_csv("model/scheme_chunks_enriched.csv")

In [182]:
# Reconstruct category lists safely from the saved CSV
def parse_categories(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

chunks_df["category_list_parsed"] = (
    chunks_df["category_list_parsed"].apply(parse_categories)
)

assert len(chunks_df) == index.ntotal
assert chunks_df["scheme_id"].notna().all()


FIELD_KEYWORDS = {
    "eligibility": [
        "eligibility", "eligible", "qualify", "qualification",
        "who can apply", "age limit", "income limit"
    ],
    "benefits": [
        "benefit", "benefits", "financial assistance", "subsidy",
        "loan", "amount", "how much", "support"
    ],
    "documents": [
        "document", "documents", "proof", "certificate",
        "required papers"
    ],
    "application": [
        "apply", "application", "how to apply",
        "register", "registration", "where to apply"
    ],
    "description": [
        "what is", "overview", "purpose", "about this scheme"
    ]
}


def infer_requested_fields(question):
    q = question.lower()
    return [
        field
        for field, keywords in FIELD_KEYWORDS.items()
        if any(keyword in q for keyword in keywords)
    ]


def retrieve_schemes(
    question,
    top_k_chunks=100,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    field_boost=0.04
):
    query_vector = encoder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    # With filters, search the entire index so relevant filtered
    # schemes are not missed by the initial global top-k cutoff.
    if level_filter is not None or category_filter is not None:
        search_k = index.ntotal
    else:
        search_k = min(top_k_chunks, index.ntotal)

    similarities, faiss_ids = index.search(query_vector, search_k)

    valid = faiss_ids[0] >= 0
    ids = faiss_ids[0][valid]

    candidates = chunks_df.iloc[ids].copy()
    candidates["faiss_id"] = ids
    candidates["similarity"] = similarities[0][valid]

    if level_filter is not None:
        candidates = candidates[
            candidates["level"].astype(str).str.casefold()
            == level_filter.casefold()
        ]

    if category_filter is not None:
        wanted = category_filter.strip().casefold()

        candidates = candidates[
            candidates["category_list_parsed"].apply(
                lambda cats: any(
                    str(cat).strip().casefold() == wanted
                    for cat in cats
                )
            )
        ]

    if candidates.empty:
        print("No schemes matched the requested filters.")
        return pd.DataFrame()

    requested_fields = infer_requested_fields(question)

    candidates["field_boost"] = np.where(
        candidates["field_type"].isin(requested_fields),
        field_boost,
        0.0
    )

    candidates["ranking_score"] = (
        candidates["similarity"] + candidates["field_boost"]
    )

    candidates = candidates.sort_values(
        "ranking_score", ascending=False
    )

    # One best matching chunk per scheme
    best_per_scheme = candidates.drop_duplicates(
        subset=["scheme_id"], keep="first"
    )

    result_cols = [
        "faiss_id", "similarity", "field_boost", "ranking_score",
        "scheme_id", "scheme_name", "level",
        "category_list_parsed", "field_type"
    ]

    return best_per_scheme[result_cols].head(
        top_k_schemes
    ).reset_index(drop=True)


In [183]:

# TEST A: Confirm the education category exists in metadata
education_count = chunks_df["category_list_parsed"].apply(
    lambda cats: "Education & Learning" in cats
).sum()

print("Education-tagged chunks:", education_count)

# TEST B: Education filter
print("\nTEST: Education category")
education_results = retrieve_schemes(
    "Which scholarships are available for college students?",
    category_filter="Education & Learning"
)
print(education_results.to_string(index=False))

# TEST C: Central filter
print("\nTEST: Central schemes")
central_results = retrieve_schemes(
    "What financial support is available for small businesses?",
    level_filter="Central"
)
print(central_results.to_string(index=False))

# TEST D: Combined filters
print("\nTEST: Central education schemes")
combined_results = retrieve_schemes(
    "Which scholarships are available for college students?",
    level_filter="Central",
    category_filter="Education & Learning"
)
print(combined_results.to_string(index=False))


Education-tagged chunks: 5619

TEST: Education category
 faiss_id  similarity  field_boost  ranking_score                scheme_id                                                                                                                      scheme_name   level                                 category_list_parsed  field_type
    18838    0.583159          0.0       0.583159         SCHEME_sheambgoc                                        Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category   State                               [Education & Learning] eligibility
    17961    0.575535          0.0       0.575535         SCHEME_sasseceie Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments   State                               [Education & Learning] eligibility
     8688    0.561615          0.0       0.561615 SCHEME_iasri-scholarship                                       

In [184]:

def needs_clarification(question):
    q = question.lower().strip()

    generic_document_questions = [
        "what documents are required to apply for a scheme",
        "what documents are required for a scheme",
        "which documents are required for a scheme",
        "what documents do i need for a scheme",
        "documents required to apply for a scheme",
        "what is required to apply for a scheme",
    ]

    generic_application_questions = [
        "how do i apply for a scheme",
        "how can i apply for a scheme",
        "how to apply for a government scheme",
        "how do i register for a scheme",
    ]

    if any(phrase in q for phrase in generic_document_questions):
        return (
            "Which government scheme are you interested in? "
            "Document requirements differ by scheme. You can share "
            "the scheme name, or tell me what support you need."
        )

    if any(phrase in q for phrase in generic_application_questions):
        return (
            "Which scheme would you like to apply for? "
            "If you don't know its name, tell me what support you need "
            "and I can help you find relevant schemes."
        )

    return None


# Test the clarification behavior
test_questions = [
    "What documents are required to apply for a scheme?",
    "How do I apply for a scheme?",
    "What documents are required for the PM scholarship?",
    "What financial assistance is available for farmers?",
]

for question in test_questions:
    print("\nQuestion:", question)
    response = needs_clarification(question)

    if response:
        print("Clarification:", response)
    else:
        print("Action: Continue to scheme retrieval")




Question: What documents are required to apply for a scheme?
Clarification: Which government scheme are you interested in? Document requirements differ by scheme. You can share the scheme name, or tell me what support you need.

Question: How do I apply for a scheme?
Clarification: Which scheme would you like to apply for? If you don't know its name, tell me what support you need and I can help you find relevant schemes.

Question: What documents are required for the PM scholarship?
Action: Continue to scheme retrieval

Question: What financial assistance is available for farmers?
Action: Continue to scheme retrieval


In [196]:

import pandas as pd
import numpy as np
import re
import ast
import faiss
from sentence_transformers import SentenceTransformer

# Load enriched chunks and scheme metadata
chunks_df = pd.read_csv("model/scheme_chunks_enriched.csv")
schemes_df = pd.read_csv("processed_schemes_v3.csv")

# Restore category lists
def parse_categories(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

if "category_list_parsed" not in chunks_df.columns:
    chunks_df["category_list_parsed"] = (
        chunks_df["category_list"].apply(parse_categories)
    )
else:
    chunks_df["category_list_parsed"] = (
        chunks_df["category_list_parsed"].apply(parse_categories)
    )

# Load FAISS index and encoder
index = faiss.read_index("model/scheme_index.faiss")

with open("model/scheme_encoder_name.txt", "r") as f:
    encoder_name = f.read().strip()

encoder = SentenceTransformer(encoder_name)

assert len(chunks_df) == index.ntotal, "Chunks and FAISS index mismatch"
assert schemes_df["scheme_id"].is_unique, "Duplicate scheme IDs"

print("Cell 1 complete")
print("Chunks:", len(chunks_df))
print("Schemes:", len(schemes_df))
print("FAISS vectors:", index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cell 1 complete
Chunks: 22068
Schemes: 3397
FAISS vectors: 22068


In [197]:

def normalize_for_matching(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


scheme_catalog = (
    schemes_df[["scheme_id", "scheme_name", "level"]]
    .drop_duplicates(subset=["scheme_id"])
    .copy()
    .reset_index(drop=True)
)

scheme_catalog["normalized_name"] = (
    scheme_catalog["scheme_name"].apply(normalize_for_matching)
)


def find_exact_scheme_name(question):
    q = normalize_for_matching(question)

    matches = scheme_catalog[
        scheme_catalog["normalized_name"].apply(
            lambda name: len(name) >= 5 and name in q
        )
    ]

    if matches.empty:
        return None

    return matches.loc[
        matches["normalized_name"].str.len().idxmax()
    ]


def get_scheme_details(scheme_id):
    matches = schemes_df[schemes_df["scheme_id"] == scheme_id]

    if matches.empty:
        return None

    row = matches.iloc[0]

    return {
        "scheme_id": row["scheme_id"],
        "scheme_name": row["scheme_name"],
        "level": row["level"],
        "categories": row["normalized_categories"],
        "eligibility": row["eligibility"],
        "benefits": row["benefits"],
        "application": row["application"],
        "documents": row["documents"],
        "details": row["details"],
    }


print("Cell 2 complete")


Cell 2 complete


In [198]:

def needs_clarification(question):
    q = normalize_for_matching(question)

    generic_document_questions = [
        "what documents are required to apply for a scheme",
        "what documents are required for a scheme",
        "which documents are required for a scheme",
        "what documents do i need for a scheme",
        "documents required to apply for a scheme",
        "what is required to apply for a scheme",
    ]

    generic_application_questions = [
        "how do i apply for a scheme",
        "how can i apply for a scheme",
        "how to apply for a government scheme",
        "how do i register for a scheme",
    ]

    if any(phrase in q for phrase in generic_document_questions):
        return (
            "Which government scheme are you interested in? "
            "Document requirements differ by scheme. You can share "
            "the scheme name, or tell me what support you need."
        )

    if any(phrase in q for phrase in generic_application_questions):
        return (
            "Which scheme would you like to apply for? If you don't "
            "know its name, tell me what support you need and I can "
            "help you find relevant schemes."
        )

    return None


def needs_scheme_clarification(question):
    q = normalize_for_matching(question)

    ambiguous_phrases = [
        "pm scholarship",
        "pm scheme",
        "prime minister scheme",
        "government scholarship",
        "farmer scheme",
        "women scheme",
        "student scholarship",
    ]

    for phrase in ambiguous_phrases:
        if phrase in q:
            # Do not ask for clarification if a full known scheme name
            # is already present in the question.
            if find_exact_scheme_name(question) is not None:
                return None

            if "scholarship" in phrase:
                return (
                    "Which scholarship do you mean? Please share its full "
                    "name, your education level, or the type of scholarship "
                    "you need. I can then narrow down the options."
                )

            return (
                "Could you tell me which scheme you mean, or describe "
                "the support you need? I'll use that information to "
                "narrow down the available schemes."
            )

    return None


print("Cell 3 complete")


Cell 3 complete


In [199]:

FIELD_KEYWORDS = {
    "eligibility": [
        "eligibility", "eligible", "qualify", "qualification",
        "who can apply", "age limit", "income limit"
    ],
    "benefits": [
        "benefit", "benefits", "financial assistance", "subsidy",
        "loan", "amount", "how much", "support"
    ],
    "documents": [
        "document", "documents", "proof", "certificate",
        "required papers"
    ],
    "application": [
        "apply", "application", "how to apply",
        "register", "registration", "where to apply"
    ],
    "description": [
        "what is", "overview", "purpose", "about this scheme"
    ]
}


def infer_requested_fields(question):
    q = question.lower()
    return [
        field
        for field, keywords in FIELD_KEYWORDS.items()
        if any(keyword in q for keyword in keywords)
    ]


def retrieve_schemes(
    question,
    top_k_chunks=100,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    field_boost=0.04
):
    query_vector = encoder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    if level_filter is not None or category_filter is not None:
        search_k = index.ntotal
    else:
        search_k = min(top_k_chunks, index.ntotal)

    similarities, faiss_ids = index.search(query_vector, search_k)

    valid = faiss_ids[0] >= 0
    ids = faiss_ids[0][valid]

    candidates = chunks_df.iloc[ids].copy()
    candidates["faiss_id"] = ids
    candidates["similarity"] = similarities[0][valid]

    if level_filter is not None:
        candidates = candidates[
            candidates["level"].astype(str).str.casefold()
            == level_filter.casefold()
        ]

    if category_filter is not None:
        wanted = category_filter.strip().casefold()

        candidates = candidates[
            candidates["category_list_parsed"].apply(
                lambda cats: any(
                    str(cat).strip().casefold() == wanted
                    for cat in cats
                )
            )
        ]

    if candidates.empty:
        return pd.DataFrame()

    requested_fields = infer_requested_fields(question)

    candidates["field_boost"] = np.where(
        candidates["field_type"].isin(requested_fields),
        field_boost,
        0.0
    )

    candidates["ranking_score"] = (
        candidates["similarity"] + candidates["field_boost"]
    )

    candidates = candidates.sort_values(
        "ranking_score", ascending=False
    )

    best_per_scheme = candidates.drop_duplicates(
        subset=["scheme_id"], keep="first"
    )

    result_cols = [
        "faiss_id", "similarity", "field_boost", "ranking_score",
        "scheme_id", "scheme_name", "level",
        "category_list_parsed", "field_type"
    ]

    return best_per_scheme[result_cols].head(
        top_k_schemes
    ).reset_index(drop=True)


print("Cell 4 complete")


Cell 4 complete


In [251]:

def answer_scheme_query(
    question,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    user_state=None
):
    # 1. Generic clarification
    clarification = needs_clarification(question)

    if clarification:
        return {
            "response_type": "clarification",
            "message": clarification,
            "scheme": None,
            "results": pd.DataFrame()
        }

    # 2. Ambiguous scheme-name clarification
    scheme_clarification = needs_scheme_clarification(question)

    if scheme_clarification:
        return {
            "response_type": "clarification",
            "message": scheme_clarification,
            "scheme": None,
            "results": pd.DataFrame()
        }


    # 3. Exact scheme-name match
    match = find_exact_scheme_name(question)

    if match is not None:
        scheme = get_scheme_details(match["scheme_id"])

        if scheme is not None:
            message = (
                f"Found the scheme: {scheme['scheme_name']}. "
                "The details below are from the dataset. "
                "Please verify the official information before applying."
            )

            # Add a state note when the user supplies their state
            if user_state:
                eligibility_text = scheme.get("eligibility", "")
                state_mentions = extract_state_mentions(eligibility_text)

                state_note = state_match_explanation(
                    state_mentions,
                    user_state
                )

                message += f" State eligibility note: {state_note}"

            return {
                "response_type": "specific_scheme",
                "message": message,
                "scheme": scheme,
                "results": pd.DataFrame()
            }


    # 4. Semantic retrieval
    results = retrieve_schemes(
        question=question,
        top_k_schemes=top_k_schemes,
        level_filter=level_filter,
        category_filter=category_filter
    )

    if results.empty:
        return {
            "response_type": "no_results",
            "message": (
                "I couldn't find matching schemes with those filters. "
                "Try describing the support you need or changing the filters."
            ),
            "scheme": None,
            "results": results
        }

    results = results.copy()

    # 5. Join eligibility fields without changing retrieval scores
    eligibility_columns = schemes_df[
        ["scheme_id", "eligibility", "details"]
    ].drop_duplicates(subset=["scheme_id"])

    results = results.merge(
        eligibility_columns,
        on="scheme_id",
        how="left",
        validate="many_to_one"
    )

    # 6. Optional state-aware explanations
    if user_state:
        results["state_mentions"] = results["eligibility"].apply(
            extract_state_mentions
        )

        results["state_explanation"] = results["state_mentions"].apply(
            lambda mentions: state_match_explanation(
                mentions, user_state
            )
        )

        message = (
            f"I found {len(results)} potentially relevant schemes. "
            f"State notes are provided for {user_state}, based on "
            "state names detected in the dataset. These notes do not "
            "confirm eligibility. Check official guidelines before applying."
        )
    else:
        message = (
            f"I found {len(results)} potentially relevant schemes. "
            "Check each scheme's eligibility and official details "
            "before applying."
        )

    return {
        "response_type": "scheme_results",
        "message": message,
        "scheme": None,
        "results": results
    }


In [252]:

test_questions = [
    "What documents are required to apply for a scheme?",
    "What documents are required for the PM scholarship?",
    "What documents are required for IASRI Scholarship For M.Sc And Ph.D?",
    "What financial assistance is available for farmers?",
    "Which scholarships are available for college students?"
]

for question in test_questions:
    response = answer_scheme_query(question)

    print("\n" + "=" * 75)
    print("QUESTION:", question)
    print("TYPE:", response["response_type"])
    print("MESSAGE:", response["message"])

    if response["scheme"] is not None:
        scheme = response["scheme"]
        print("SCHEME:", scheme["scheme_name"])

        if "documents" in question.lower():
            print("DOCUMENTS:", scheme["documents"])

    if not response["results"].empty:
        print(
            response["results"][
                ["scheme_name", "level", "similarity", "field_type"]
            ].to_string(index=False)
        )



QUESTION: What documents are required to apply for a scheme?
TYPE: clarification
MESSAGE: Which government scheme are you interested in? Document requirements differ by scheme. You can share the scheme name, or tell me what support you need.

QUESTION: What documents are required for the PM scholarship?
TYPE: clarification
MESSAGE: Which scholarship do you mean? Please share its full name, your education level, or the type of scholarship you need. I can then narrow down the options.

QUESTION: What documents are required for IASRI Scholarship For M.Sc And Ph.D?
TYPE: specific_scheme
MESSAGE: Found the scheme: IASRI Scholarship For M.Sc And Ph.D. The details below are from the dataset. Please verify the official information before applying.
SCHEME: IASRI Scholarship For M.Sc And Ph.D
DOCUMENTS: Required Documents : Upload Image ( less than 100 kb and greater than10 kb). Signature (less than 20 kb and greater than 10 kb). Documents (All the requisite documents starting from Matriculatio

In [253]:

# Inspect target-group clues in scheme names and eligibility text

target_groups = {
    "college_students": [
        "college student", "college students", "undergraduate",
        "postgraduate", "post graduate", "university student",
        "higher education", "degree student"
    ],
    "school_students": [
        "school student", "school students", "pre matric",
        "pre-matric", "school children"
    ],
    "farmers": [
        "farmer", "farmers", "agriculturalist", "cultivator"
    ],
    "senior_citizens": [
        "senior citizen", "senior citizens", "old age", "elderly"
    ],
    "women": [
        "women", "woman", "girl child", "female beneficiary"
    ],
    "unemployed": [
        "unemployed", "unemployment", "job seeker"
    ]
}

# Count matching scheme records using scheme name and eligibility
for group, keywords in target_groups.items():
    mask = schemes_df.apply(
        lambda row: any(
            keyword in (
                str(row.get("scheme_name", "")) + " " +
                str(row.get("eligibility", ""))
            ).lower()
            for keyword in keywords
        ),
        axis=1
    )

    print(f"{group}: {int(mask.sum())} matching schemes")

# Inspect scholarship examples for manual relevance checking
scholarship_mask = schemes_df["scheme_name"].str.contains(
    "scholarship", case=False, na=False
)

scholarship_examples = schemes_df.loc[
    scholarship_mask,
    ["scheme_name", "level", "eligibility"]
].head(15)

print("\nScholarship examples:")
print(scholarship_examples.to_string(index=False))


college_students: 176 matching schemes
school_students: 43 matching schemes
farmers: 303 matching schemes
senior_citizens: 90 matching schemes
women: 390 matching schemes
unemployed: 81 matching schemes

Scholarship examples:
                                                                                                                                                                                    scheme_name   level                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [254]:

TARGET_GROUP_KEYWORDS = {
    "college_students": [
        "college student", "college students", "undergraduate",
        "postgraduate", "post graduate", "higher education",
        "degree level", "degree course", "professional course",
        "university student"
    ],
    "school_students": [
        "school student", "school students", "pre matric",
        "pre-matric", "class 6", "class 10", "school children"
    ],
    "farmers": [
        "farmer", "farmers", "agriculture", "cultivator",
        "agricultural land", "agricultural activities"
    ],
    "senior_citizens": [
        "senior citizen", "senior citizens", "old age", "elderly"
    ],
    "women": [
        "women", "woman", "girl student", "female beneficiary",
        "maternity", "widow"
    ],
    "unemployed": [
        "unemployed", "unemployment", "job seeker",
        "employment oriented", "skill development"
    ]
}


def infer_target_groups(question):
    q = normalize_for_matching(question)
    detected = []

    group_triggers = {
        "college_students": [
            "college student", "college students", "university student",
            "undergraduate", "postgraduate", "higher education"
        ],
        "school_students": [
            "school student", "school students", "school children"
        ],
        "farmers": ["farmer", "farmers", "farming"],
        "senior_citizens": [
            "senior citizen", "senior citizens", "elderly", "old age"
        ],
        "women": ["women", "woman", "maternity", "widow"],
        "unemployed": ["unemployed", "job seeker"]
    }

    for group, triggers in group_triggers.items():
        if any(trigger in q for trigger in triggers):
            detected.append(group)

    return detected


def target_group_score(question, scheme_name, eligibility, details=""):
    groups = infer_target_groups(question)

    if not groups:
        return 0.0, []

    text = normalize_for_matching(
        f"{scheme_name} {eligibility} {details}"
    )

    matched_groups = []
    score = 0.0

    for group in groups:
        keywords = TARGET_GROUP_KEYWORDS[group]

        if any(
            normalize_for_matching(keyword) in text
            for keyword in keywords
        ):
            matched_groups.append(group)
            score += 0.05

    return score, matched_groups


def retrieve_schemes_with_target_relevance(
    question,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    target_boost=0.05
):
    # Start with the existing semantic retrieval
    results = retrieve_schemes(
        question=question,
        top_k_schemes=100,
        level_filter=level_filter,
        category_filter=category_filter
    )

    if results.empty:
        return results

    # Join scheme-level eligibility and details
    extra = schemes_df[
        ["scheme_id", "eligibility", "details"]
    ].copy()

    results = results.merge(
        extra,
        on="scheme_id",
        how="left",
        validate="one_to_one"
    )

    scores = results.apply(
        lambda row: target_group_score(
            question,
            row["scheme_name"],
            row["eligibility"],
            row["details"]
        ),
        axis=1
    )

    results["target_boost"] = scores.apply(lambda item: item[0])
    results["matched_target_groups"] = scores.apply(
        lambda item: ", ".join(item[1])
    )

    results["final_score"] = (
        results["ranking_score"] + results["target_boost"]
    )

    results = results.sort_values(
        "final_score", ascending=False
    ).head(top_k_schemes)

    return results.reset_index(drop=True)


print("Step 21 setup complete.")
print("Target groups:", list(TARGET_GROUP_KEYWORDS.keys()))


Step 21 setup complete.
Target groups: ['college_students', 'school_students', 'farmers', 'senior_citizens', 'women', 'unemployed']


In [255]:

question = "Which scholarships are available for college students?"

results = retrieve_schemes_with_target_relevance(
    question,
    top_k_schemes=10,
    category_filter="Education & Learning"
)

print("Question:", question)
print(
    results[
        [
            "scheme_name",
            "level",
            "similarity",
            "ranking_score",
            "target_boost",
            "final_score",
            "matched_target_groups",
            "field_type"
        ]
    ].to_string(index=False)
)


Question: Which scholarships are available for college students?
                                                                                                                     scheme_name   level  similarity  ranking_score  target_boost  final_score matched_target_groups  field_type
                                       Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category   State    0.583159       0.583159          0.05     0.633159      college_students eligibility
Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments   State    0.575535       0.575535          0.05     0.625535      college_students eligibility
                                                                                             IASRI Scholarship For M.Sc And Ph.D Central    0.561615       0.561615          0.05     0.611615      college_students    benefits
                                   

In [256]:
print("results columns:")
print(results.columns.tolist())

print("\nschemes_df columns:")
print(schemes_df.columns.tolist())

results columns:
['faiss_id', 'similarity', 'field_boost', 'ranking_score', 'scheme_id', 'scheme_name', 'level', 'category_list_parsed', 'field_type', 'eligibility', 'details', 'target_boost', 'matched_target_groups', 'final_score']

schemes_df columns:
['source_row_id', 'scheme_name', 'slug', 'details', 'benefits', 'eligibility', 'application', 'documents', 'level', 'schemeCategory', 'tags', 'scheme_id', 'retrieval_text', 'category_list', 'category_unrecognized', 'normalized_categories']


In [257]:
def classify_education_relevance(scheme_name, eligibility, details=""):
    text = normalize_for_matching(
        f"{scheme_name} {eligibility} {details}"
    )

    college_indicators = [
        "undergraduate", "postgraduate", "post graduate",
        "higher education", "degree level", "degree course",
        "professional course", "college student", "college students",
        "m sc", "ph d", "university"
    ]

    school_indicators = [
        "pre matric", "class 1", "class 2", "class 3",
        "class 4", "class 5", "class 6", "class 7",
        "class 8", "class 9", "class 10", "class 11",
        "class 12", "school student", "school students",
        "school children"
    ]

    college_found = any(term in text for term in college_indicators)
    school_found = any(term in text for term in school_indicators)

    if college_found:
        return "college_relevant"
    elif school_found:
        return "school_only"
    else:
        return "uncertain"


# Use the columns already present in results
inspection = results.copy()

inspection["education_relevance"] = inspection.apply(
    lambda row: classify_education_relevance(
        row["scheme_name"],
        row["eligibility"],
        row["details"]
    ),
    axis=1
)

print(
    inspection[
        [
            "scheme_name",
            "education_relevance",
            "matched_target_groups",
            "eligibility"
        ]
    ].to_string(index=False)
)

                                                                                                                     scheme_name education_relevance matched_target_groups                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [258]:
def classify_education_relevance(scheme_name, eligibility, details=""):
    text = normalize_for_matching(
        f"{scheme_name} {eligibility} {details}"
    )

    college_indicators = [
        "undergraduate", "postgraduate", "post graduate",
        "higher education", "degree course", "professional course",
        "college student", "college students", "m sc", "ph d",
        "university", "class 11th 12th college",
        "post matric", "graduate degree"
    ]

    school_indicators = [
        "pre matric", "school student", "school students",
        "school children", "class 1", "class 2", "class 3",
        "class 4", "class 5", "class 6", "class 7",
        "class 8", "class 9", "class 10", "class 11",
        "class 12"
    ]

    college_found = any(term in text for term in college_indicators)
    school_found = any(term in text for term in school_indicators)

    if college_found and school_found:
        return "school_and_college"
    elif college_found:
        return "college_relevant"
    elif school_found:
        return "school_relevant"
    else:
        return "uncertain"


inspection = results.copy()

inspection["education_relevance"] = inspection.apply(
    lambda row: classify_education_relevance(
        row["scheme_name"],
        row["eligibility"],
        row["details"]
    ),
    axis=1
)

print(
    inspection[
        ["scheme_name", "education_relevance", "eligibility"]
    ].to_string(index=False)
)

print("\nLabel counts:")
print(inspection["education_relevance"].value_counts())

                                                                                                                     scheme_name education_relevance                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [259]:
import re

STATE_NAMES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chhattisgarh", "Goa", "Gujarat", "Haryana",
    "Himachal Pradesh", "Jharkhand", "Karnataka", "Kerala",
    "Madhya Pradesh", "Maharashtra", "Manipur", "Meghalaya",
    "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura",
    "Uttar Pradesh", "Uttarakhand", "West Bengal",
    "Delhi", "Jammu and Kashmir"
]

def extract_state_mentions(text):
    text = str(text)
    found = [
        state for state in STATE_NAMES
        if re.search(
            r"\b" + re.escape(state) + r"\b",
            text,
            flags=re.IGNORECASE
        )
    ]
    return found

inspection["state_mentions"] = inspection["eligibility"].apply(
    extract_state_mentions
)

print(
    inspection[
        ["scheme_name", "state_mentions"]
    ].to_string(index=False)
)

print("\nState mention counts:")
print(
    inspection["state_mentions"]
    .explode()
    .dropna()
    .value_counts()
)

                                                                                                                     scheme_name   state_mentions
                                       Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category    [Maharashtra]
Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments            [Goa]
                                                                                             IASRI Scholarship For M.Sc And Ph.D          [Delhi]
                                                        Post-Graduate Scholarships for Professional Courses for SC/ST Candidates               []
                                                                                          Post Matric Scholarship For Minorities               []
                    Free Education Scholarship for Professional Courses (Engineering, Medical, Agriculture, Veterinary, and 

In [260]:
def classify_state_restriction(state_mentions):
    if state_mentions:
        return "state_mentioned_check_restriction"
    return "no_state_mentioned_check_source"


inspection["state_restriction_status"] = (
    inspection["state_mentions"].apply(classify_state_restriction)
)

print(
    inspection[
        [
            "scheme_name",
            "state_mentions",
            "state_restriction_status"
        ]
    ].to_string(index=False)
)

print("\nRestriction flag counts:")
print(inspection["state_restriction_status"].value_counts())

                                                                                                                     scheme_name   state_mentions          state_restriction_status
                                       Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category    [Maharashtra] state_mentioned_check_restriction
Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments            [Goa] state_mentioned_check_restriction
                                                                                             IASRI Scholarship For M.Sc And Ph.D          [Delhi] state_mentioned_check_restriction
                                                        Post-Graduate Scholarships for Professional Courses for SC/ST Candidates               []   no_state_mentioned_check_source
                                                                                          Post Matri

In [261]:
USER_STATE = "Tamil Nadu"

def assess_state_match(state_mentions, user_state):
    if not state_mentions:
        return "unknown_needs_verification"

    if any(
        state.lower() == user_state.lower()
        for state in state_mentions
    ):
        return "state_matches_user"

    return "different_state_mentioned_check"


inspection["state_match"] = inspection["state_mentions"].apply(
    lambda mentions: assess_state_match(mentions, USER_STATE)
)

print(
    inspection[
        [
            "scheme_name",
            "state_mentions",
            "state_match"
        ]
    ].to_string(index=False)
)

print("\nState-match counts:")
print(inspection["state_match"].value_counts())

                                                                                                                     scheme_name   state_mentions                     state_match
                                       Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category    [Maharashtra] different_state_mentioned_check
Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments            [Goa] different_state_mentioned_check
                                                                                             IASRI Scholarship For M.Sc And Ph.D          [Delhi] different_state_mentioned_check
                                                        Post-Graduate Scholarships for Professional Courses for SC/ST Candidates               []      unknown_needs_verification
                                                                                          Post Matric Scholars

In [262]:
def state_match_explanation(state_mentions, user_state):
    if not state_mentions:
        return (
            "State eligibility is unconfirmed. "
            "Check the official scheme guidelines."
        )

    matching_states = [
        state for state in state_mentions
        if state.lower() == user_state.lower()
    ]

    if matching_states:
        return (
            f"{user_state} is mentioned in the eligibility text. "
            "Verify the exact residence and other requirements."
        )

    mentioned = ", ".join(state_mentions)

    return (
        f"The eligibility text mentions {mentioned}, not "
        f"{user_state}. Check the official guidelines to confirm "
        "whether applicants from your state can apply."
    )


inspection["state_explanation"] = inspection["state_mentions"].apply(
    lambda mentions: state_match_explanation(mentions, USER_STATE)
)

print(
    inspection[
        [
            "scheme_name",
            "state_match",
            "state_explanation"
        ]
    ].to_string(index=False)
)

                                                                                                                     scheme_name                     state_match                                                                                                                                    state_explanation
                                       Scholarships For Higher Education Abroad To Meritorious Boys And Girls From Open Category different_state_mentioned_check    The eligibility text mentions Maharashtra, not Tamil Nadu. Check the official guidelines to confirm whether applicants from your state can apply.
Scheme for Award of Scholarships to School or College-Going Children of Employees/Industrial Employees in Various Establishments different_state_mentioned_check            The eligibility text mentions Goa, not Tamil Nadu. Check the official guidelines to confirm whether applicants from your state can apply.
                                                                      

In [263]:
import inspect

print(inspect.getsource(answer_scheme_query))

def answer_scheme_query(
    question,
    top_k_schemes=5,
    level_filter=None,
    category_filter=None,
    user_state=None
):
    # 1. Generic clarification
    clarification = needs_clarification(question)

    if clarification:
        return {
            "response_type": "clarification",
            "message": clarification,
            "scheme": None,
            "results": pd.DataFrame()
        }

    # 2. Ambiguous scheme-name clarification
    scheme_clarification = needs_scheme_clarification(question)

    if scheme_clarification:
        return {
            "response_type": "clarification",
            "message": scheme_clarification,
            "scheme": None,
            "results": pd.DataFrame()
        }

    
    # 3. Exact scheme-name match
    match = find_exact_scheme_name(question)

    if match is not None:
        scheme = get_scheme_details(match["scheme_id"])

        if scheme is not None:
            message = (
                f"Found the scheme: {sch

In [264]:
print("Returned columns:")
print(test_answer["results"].columns.tolist())

print("\nFirst result:")
display(test_answer["results"].head(1))

Returned columns:
['faiss_id', 'similarity', 'field_boost', 'ranking_score', 'scheme_id', 'scheme_name', 'level', 'category_list_parsed', 'field_type', 'eligibility', 'details', 'state_mentions', 'state_explanation']

First result:


,faiss_id,similarity,field_boost,ranking_score,scheme_id,scheme_name,level,category_list_parsed,field_type,eligibility,details,state_mentions,state_explanation
0,18838,0.583159,0.0,0.583159,SCHEME_sheambgoc,Scholarships For Higher Education Abroad To Me...,State,[Education & Learning],eligibility,The candidate and the candidate's mother/fathe...,The scheme “Scholarships for Higher Education ...,[Maharashtra],"The eligibility text mentions Maharashtra, not..."


In [265]:
test_answer = answer_scheme_query(
    question="Which scholarships are available for college students?",
    top_k_schemes=5,
    category_filter="Education & Learning",
    user_state="Tamil Nadu"
)

print("Response type:", test_answer["response_type"])
print("Message:", test_answer["message"])

if not test_answer["results"].empty:
    display(
        test_answer["results"][
            [
                "scheme_name",
                "state_mentions",
                "state_explanation"
            ]
        ]
    )

Response type: scheme_results
Message: I found 5 potentially relevant schemes. State notes are provided for Tamil Nadu, based on state names detected in the dataset. These notes do not confirm eligibility. Check official guidelines before applying.


,scheme_name,state_mentions,state_explanation
0,Scholarships For Higher Education Abroad To Me...,[Maharashtra],"The eligibility text mentions Maharashtra, not..."
1,Scheme for Award of Scholarships to School or ...,[Goa],"The eligibility text mentions Goa, not Tamil N..."
2,IASRI Scholarship For M.Sc And Ph.D,[Delhi],"The eligibility text mentions Delhi, not Tamil..."
3,Pre Matric Scholarship For Minorities,[],State eligibility is unconfirmed. Check the of...
4,Post-Graduate Scholarships for Professional Co...,[],State eligibility is unconfirmed. Check the of...


In [266]:
test_cases = [
    (["Tamil Nadu"], "Tamil Nadu"),
    (["Madhya Pradesh"], "Tamil Nadu"),
    ([], "Tamil Nadu"),
]

for mentions, user_state in test_cases:
    print("Detected states:", mentions)
    print("User state:", user_state)
    print(
        "Explanation:",
        state_match_explanation(mentions, user_state)
    )
    print("-" * 70)

Detected states: ['Tamil Nadu']
User state: Tamil Nadu
Explanation: Tamil Nadu is mentioned in the eligibility text. Verify the exact residence and other requirements.
----------------------------------------------------------------------
Detected states: ['Madhya Pradesh']
User state: Tamil Nadu
Explanation: The eligibility text mentions Madhya Pradesh, not Tamil Nadu. Check the official guidelines to confirm whether applicants from your state can apply.
----------------------------------------------------------------------
Detected states: []
User state: Tamil Nadu
Explanation: State eligibility is unconfirmed. Check the official scheme guidelines.
----------------------------------------------------------------------


In [267]:
# Test 1: General scholarship query with state context
general_test = answer_scheme_query(
    question="Which scholarships are available for college students?",
    top_k_schemes=5,
    category_filter="Education & Learning",
    user_state="Tamil Nadu"
)

print("TEST 1 — General scholarship query")
print("Type:", general_test["response_type"])
print("Message:", general_test["message"])

if not general_test["results"].empty:
    display(
        general_test["results"][
            [
                "scheme_name",
                "state_mentions",
                "state_explanation"
            ]
        ]
    )

print("\n" + "=" * 80 + "\n")

# Test 2: Exact scheme-name query
exact_test = answer_scheme_query(
    question="IASRI Scholarship For M.Sc And Ph.D",
    user_state="Tamil Nadu"
)

print("TEST 2 — Exact scheme-name query")
print("Type:", exact_test["response_type"])
print("Message:", exact_test["message"])

if exact_test["scheme"] is not None:
    print("Scheme:", exact_test["scheme"]["scheme_name"])
    print("Eligibility:", exact_test["scheme"]["eligibility"])

TEST 1 — General scholarship query
Type: scheme_results
Message: I found 5 potentially relevant schemes. State notes are provided for Tamil Nadu, based on state names detected in the dataset. These notes do not confirm eligibility. Check official guidelines before applying.


,scheme_name,state_mentions,state_explanation
0,Scholarships For Higher Education Abroad To Me...,[Maharashtra],"The eligibility text mentions Maharashtra, not..."
1,Scheme for Award of Scholarships to School or ...,[Goa],"The eligibility text mentions Goa, not Tamil N..."
2,IASRI Scholarship For M.Sc And Ph.D,[Delhi],"The eligibility text mentions Delhi, not Tamil..."
3,Pre Matric Scholarship For Minorities,[],State eligibility is unconfirmed. Check the of...
4,Post-Graduate Scholarships for Professional Co...,[],State eligibility is unconfirmed. Check the of...




TEST 2 — Exact scheme-name query
Type: specific_scheme
Message: Found the scheme: IASRI Scholarship For M.Sc And Ph.D. The details below are from the dataset. Please verify the official information before applying. State eligibility note: The eligibility text mentions Delhi, not Tamil Nadu. Check the official guidelines to confirm whether applicants from your state can apply.
Scheme: IASRI Scholarship For M.Sc And Ph.D
Eligibility: Eligibility requirements for M.Sc. (Agricultural Statistics) Bachelor of Science degree in Agriculture / Horticulture / Forestry / Agroforestry / Sericulture / Agricultural Marketing/ B.Sc.(10+2+3 System) with 60% marks or its equivalent Overall grade point average (OGPA) for the general category, and 55% marks or its equivalent OGPA for SC/ST/PH candidates. ﻿ The minimum residential requirement for both M.Sc. degrees is 2 academic years from the date of admission. The maximum time limit for completion of the M.Sc. programme of studies is 4 years from the 

In [268]:
exact_test = answer_scheme_query(
    question="IASRI Scholarship For M.Sc And Ph.D",
    user_state="Tamil Nadu"
)

print("Type:", exact_test["response_type"])
print("Message:", exact_test["message"])

Type: specific_scheme
Message: Found the scheme: IASRI Scholarship For M.Sc And Ph.D. The details below are from the dataset. Please verify the official information before applying. State eligibility note: The eligibility text mentions Delhi, not Tamil Nadu. Check the official guidelines to confirm whether applicants from your state can apply.


In [269]:
test_cases = [
    {
        "question": "What documents are required to apply for a scheme?",
        "expected": "clarification"
    },
    {
        "question": "What documents are required for the PM scholarship?",
        "expected": "clarification"
    },
    {
        "question": "IASRI Scholarship For M.Sc And Ph.D",
        "expected": "specific_scheme"
    },
    {
        "question": "Which scholarships are available for college students?",
        "expected": "scheme_results"
    }
]

for case in test_cases:
    answer = answer_scheme_query(
        question=case["question"],
        top_k_schemes=5,
        category_filter="Education & Learning",
        user_state="Tamil Nadu"
    )

    passed = answer["response_type"] == case["expected"]

    print(f"Question: {case['question']}")
    print(f"Expected: {case['expected']}")
    print(f"Actual:   {answer['response_type']}")
    print(f"Status:   {'PASS' if passed else 'FAIL'}")
    print("-" * 70)

Question: What documents are required to apply for a scheme?
Expected: clarification
Actual:   clarification
Status:   PASS
----------------------------------------------------------------------
Question: What documents are required for the PM scholarship?
Expected: clarification
Actual:   clarification
Status:   PASS
----------------------------------------------------------------------
Question: IASRI Scholarship For M.Sc And Ph.D
Expected: specific_scheme
Actual:   specific_scheme
Status:   PASS
----------------------------------------------------------------------
Question: Which scholarships are available for college students?
Expected: scheme_results
Actual:   scheme_results
Status:   PASS
----------------------------------------------------------------------


In [270]:
from pathlib import Path

frontend_path = Path("/content/drive/MyDrive/SEM-6/Mini-Project/HomePage.tsx")

if frontend_path.exists():
    frontend_code = frontend_path.read_text(encoding="utf-8")
    print("Frontend file found.")
    print("Total lines:", len(frontend_code.splitlines()))

    for i, line in enumerate(frontend_code.splitlines(), start=1):
        if any(
            term in line.lower()
            for term in [
                "scheme_results",
                "scheme_name",
                "eligibility",
                "response_type",
                "results"
            ]
        ):
            start = max(1, i - 4)
            end = min(len(frontend_code.splitlines()), i + 8)
            print(f"\n--- Lines {start}-{end} ---")
            for j in range(start, end + 1):
                print(f"{j}: {frontend_code.splitlines()[j - 1]}")
else:
    print("HomePage.tsx was not found at:", frontend_path)

HomePage.tsx was not found at: /content/drive/MyDrive/SEM-6/Mini-Project/HomePage.tsx


In [271]:
from pathlib import Path

# Check common locations for an uploaded or downloaded frontend file
possible_paths = [
    Path("/content/HomePage.tsx"),
    Path("/mnt/data/HomePage.tsx"),
]

for path in possible_paths:
    print(path, "->", "FOUND" if path.exists() else "not found")

/content/HomePage.tsx -> not found
/mnt/data/HomePage.tsx -> not found


In [272]:
import os
import zipfile
from google.colab import files

model_dir = "/content/model"

required_files = [
    "scheme_chunks.csv",
    "scheme_chunks_enriched.csv",
    "scheme_embeddings.npy",
    "scheme_encoder_name.txt",
]

zip_path = "/content/colab_retrieval_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in required_files:
        path = os.path.join(model_dir, filename)
        if not os.path.exists(path):
            print(f"Missing: {path}")
            continue
        zf.write(path, arcname=filename)
        print(f"Added: {filename}")

files.download(zip_path)

Added: scheme_chunks.csv
Added: scheme_chunks_enriched.csv
Added: scheme_embeddings.npy
Added: scheme_encoder_name.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>